# Hungarian Election Prediction 2026 | Andrási Kristóf - notebook 12


### Imports

This block loads the packages used in the notebook. The structure follows notebook 01 so the notebook opening stays consistent across the final folder.


In [1]:
# this block loads the packages used in the notebook.
import importlib
import subprocess
import sys

def ensure_packages(packages):
    # install a package only if it is missing
    for pkg in packages:
        try:
            importlib.import_module(pkg)
        except ImportError:
            print(f"{pkg} not found, installing...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
            print(f"{pkg} installed.")

ensure_packages(["pandas", "numpy", "openpyxl", "matplotlib", "re", "xlrd", "scipy", "statsmodels", "IPython"])

import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import re
import numpy as np


### Working Directory

This block sets the main project paths so file loading works the same way each time. The layout follows notebook 01.


In [2]:
# this block sets the main project paths so file loading works the same way each time.
def find_project_root(start_path=None):
    start_path = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "scripts" / "final_final_scripts").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root from the current working directory.")

here = Path.cwd().resolve()
project_root = find_project_root(here)
data_dir = project_root / "data" / "TokaGabor"

print("here:", here)
print("project_root:", project_root)
print("data_dir:", data_dir)


here: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions/scripts/final_final_scripts
project_root: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions
data_dir: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions/data/TokaGabor


Interpretation. The printed paths should point to this election project. If they do not, the later loading steps can fail.


## Notebook 12

This section keeps notebook 12 with the fixed alpha LOEO validation and the added downstream checks.


## Imports and Constants

This block loads the packages used in the notebook.


In [3]:
# this block loads the packages used in the notebook.
from pathlib import Path

import os

import warnings

import json



os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl")  # avoid matplotlib cache issues



import matplotlib

matplotlib.use("Agg")  # use non-interactive backend for notebook rendering

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd



# Monkeypatch for scipy/statsmodels compatibility issue with Python 3.13

try:

    import scipy._lib._util

    if not hasattr(scipy._lib._util, '_lazywhere'):

        import scipy.special as special

        def _lazywhere(cond, *args, f=None, fillvalue=None, **kwargs):

            output = np.empty(np.broadcast(*args, **kwargs).shape, dtype=float)

            output[~cond] = fillvalue

            output[cond] = f(*[np.asarray(a)[cond] for a in args], **kwargs)

            return output

        scipy._lib._util._lazywhere = _lazywhere

except Exception:

    pass



import statsmodels.api as sm  # for mixed-effects models

from IPython.display import display



warnings.filterwarnings("ignore")  # suppress noisy convergence warnings

np.random.seed(42)  # set random seed for reproducibility

pd.set_option("display.max_columns", 200)  # show all columns in wide tables

pd.set_option("display.width", 200)

plt.style.use("seaborn-v0_8-whitegrid")  # clean plot style



# Define the four political blocks used throughout the notebook

block_cols = ["gov", "opp", "opp_radical", "other"]

# The first three are modelled in log-ratio space; "other" is the baseline

nonbase_blocks = ["gov", "opp", "opp_radical"]

composition_floor = 1e-4  # minimum share to avoid log(0)

full_history_min_nonmissing_share = 0.95  # features must have 95% non-missing to be used



# Color scheme for the four blocks in all plots

block_colors = {

    "gov": "#f28e2b",       # orange for government block

    "opp": "#4e79a7",       # blue for main opposition

    "opp_radical": "#59a14f", # green for radical opposition

    "other": "#9c9c9c",      # grey for other/minor parties

}



def find_project_root(start_path=None):
    start_path = Path.cwd().resolve() if start_path is None else Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "scripts" / "final_final_scripts").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not find project root from the current working directory.")

here = Path.cwd().resolve()
project_root = find_project_root(here)
created_dir = project_root / "data" / "created"



selected_single_media_outlet = "pro_gov_avg"

selected_media_window = "w3"

selected_media_metric = "real_users"

selected_media_outlets = [

    "24_b",

    "borsonline_b",

    "hirado_b",

    "magyarnemzet_b",

    "mandiner_b",

    "mediaklikk_b",

    "origo_b",

    "ripost_b",

    "tv2_b",

    "tenyek_b",

]

selected_media_raw_cols = [

    f"{selected_media_metric}_{selected_media_window}_{outlet}"

    for outlet in selected_media_outlets

]

selected_media_feature_cols = ["pro_gov_avg"]

screened_feature_config = {}

screened_full_history_structure = [

    "log_pop_total",

    "young_share",

    "old_share",

    "unemployed_per_capita",

    "unemployed_univ_per_capita",

    "budget_balance_per_capita",

    "cars_per_capita",

    "cars_first_reg_per_capita",

    "doctor_per_capita",

    "dwellings_built_private_per_capita",

]

preliminary_fundamentals_models = {

    "model_1_demography_only": ["log_pop_total", "young_share", "old_share"],

    "model_2_plus_core_economy": [

        "log_pop_total",

        "young_share",

        "old_share",

        "unemployed_per_capita",

        "unemployed_univ_per_capita",

        "budget_balance_per_capita",

    ],

    "model_3_plus_modernization": [

        "log_pop_total",

        "young_share",

        "old_share",

        "unemployed_per_capita",

        "unemployed_univ_per_capita",

        "budget_balance_per_capita",

        "cars_per_capita",

        "cars_first_reg_per_capita",

    ],

    "model_4_plus_public_service_housing": [

        "log_pop_total",

        "young_share",

        "old_share",

        "unemployed_per_capita",

        "unemployed_univ_per_capita",

        "budget_balance_per_capita",

        "cars_per_capita",

        "cars_first_reg_per_capita",

        "doctor_per_capita",

        "dwellings_built_private_per_capita",

    ],

}



# LOEO validation grid parameters

LOEO_ELECTIONS = [(2018, "nat"), (2019, "ep"), (2022, "nat"), (2024, "ep")]

FIXED_POLL_RIDGE_ALPHA = 20.0    # hardcoded: MAE curve flat across all tested alphas
FIXED_PRIOR_ALPHA = 100.0         # hardcoded: MAE curve flat across all tested alphas
FIXED_LEAN_ALPHA = 30.0           # hardcoded: MAE curve flat across all tested alphas

FIXED_POLL_WEIGHT = 0.70  # fixed from literature (Linzer 2013, Economist 2020), never optimized

mae_tol = 5e-4

COND_NUMBER_THRESHOLD = 1e8  # ridge system condition numbers above this trigger alpha escalation

recent_correction_alpha = 20.0  # hardcoded for recent correction layer
house_effects_numeric_features = ["days_to_election", "undecided", "is_biztos"]
house_effects_cat_specs = [("mode", 5), ("election_type", 1)]
rw_penalty_const = 0.01  # fixed scaling for measurement variance in RW filter

print("project_root:", project_root)

print("created_dir:", created_dir)

project_root: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions
created_dir: /Users/andrasikristof/Documents/Egyetem/2026:27 - 2. félév/Election_predictions/data/created


## Load Data

This block loads the prepared tables used in the notebook.


In [4]:
# this block loads the prepared tables used in the notebook.
polls = pd.read_csv(created_dir / "polls.csv")  # national poll data

final_statistics_oevk = pd.read_csv(created_dir / "statistics.csv")  # district demographics and media

final_results_oevk_party = pd.read_csv(created_dir / "oevk_results.csv")  # district election results

mandates_all_combined = pd.read_csv(created_dir / "global_results.csv")  # official seat counts

election_assumption_2026 = pd.Timestamp("2026-04-12")  # assumed date for the 2026 election


polls["Vég"] = pd.to_datetime(polls["Vég"], errors="coerce")



table_overview = pd.DataFrame(

    {

        "table": [

            "polls",

            "final_statistics_oevk",

            "final_results_oevk_party",

            "mandates_all_combined",

        ],

        "rows": [

            len(polls),

            len(final_statistics_oevk),

            len(final_results_oevk_party),

            len(mandates_all_combined),

        ],

        "columns": [

            polls.shape[1],

            final_statistics_oevk.shape[1],

            final_results_oevk_party.shape[1],

            mandates_all_combined.shape[1],

        ],

    }

)

display(table_overview)

,table,rows,columns
0,polls,1790,40
1,final_statistics_oevk,530,308
2,final_results_oevk_party,3710,24
3,mandates_all_combined,35,10


Interpretation. This output is a quick diagnostic check of scale, spread and completeness.


Checking the 2024 ep election poll coverage without filtering any rows out.


In [5]:
# checking the 2024 ep election poll coverage without filtering any rows out
len(polls)

# convert the column to datetime if needed
#polls["Kezdet"] = pd.to_datetime(polls["Kezdet"])

# example filter logic kept here as a reference only, it is not applied in this notebook
#polls = polls[~polls["Kezdet"].between("2023-03-24", "2024-03-31")]

1790

Interpretation. This output is a quick diagnostic check of scale, spread and completeness.


This block prepares the next part of the notebook.


In [6]:
# this block prepares the next part of the notebook.
len(polls)

1790

Interpretation. This output is a quick diagnostic check of scale, spread and completeness.


## Helper Functions

This block defines helper functions that later cells reuse.


In [7]:
# this block defines helper functions that later cells reuse.
def clean_num(series):

    series = series.astype("string")

    series = series.str.replace("%", "", regex=False)

    series = series.str.replace(",", ".", regex=False)

    series = series.str.replace(r"[^0-9\.\-]", "", regex=True)

    series = series.replace("", pd.NA)

    return pd.to_numeric(series, errors="coerce").astype("float64")



def close_comp(df, cols, eps=composition_floor):

    # Make vote shares valid: clip to minimum, then normalize so they sum to 1

    arr = df[cols].astype("float64").to_numpy()

    arr = np.where(np.isfinite(arr), arr, np.nan)

    arr = np.nan_to_num(arr, nan=0.0)

    arr = np.clip(arr, eps, None)

    sums = arr.sum(axis=1, keepdims=True)

    sums = np.where(sums <= 0, 1.0, sums)

    arr = arr / sums

    return pd.DataFrame(arr, columns=cols, index=df.index)



def alr(df, cols, baseline="other"):

    # Transform vote shares to log-ratio space: log(share_i / share_baseline)

    comp = close_comp(df, cols)

    base = comp[baseline].to_numpy(dtype="float64")[:, None]

    arr = np.log(comp[[c for c in cols if c != baseline]].to_numpy(dtype="float64") / base)

    return pd.DataFrame(arr, columns=[f"alr_{c}" for c in cols if c != baseline], index=df.index)



def inverse_alr(arr, cols, baseline="other"):

    # Transform back from log-ratio space to normal vote shares

    arr = np.asarray(arr, dtype="float64")

    if arr.ndim == 1:

        arr = arr[None, :]

    arr = np.clip(arr, -30.0, 30.0)

    exp_arr = np.exp(arr)

    denom = 1.0 + exp_arr.sum(axis=1, keepdims=True)

    nonbase = exp_arr / denom

    base = 1.0 / denom

    out = np.concatenate([nonbase, base], axis=1)

    return pd.DataFrame(out, columns=[c for c in cols if c != baseline] + [baseline])



def safe_weighted_average(values, weights):

    # Compute weighted average, ignoring missing values and zero weights

    values = np.asarray(values, dtype="float64")

    weights = np.asarray(weights, dtype="float64")

    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)

    if mask.sum() == 0:

        return float(np.nanmean(values)) if np.isfinite(values).any() else np.nan

    return float(np.average(values[mask], weights=weights[mask]))



def safe_weighted_variance(values, weights):

    values = np.asarray(values, dtype="float64")

    weights = np.asarray(weights, dtype="float64")

    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)

    if mask.sum() == 0:

        return np.nan

    values = values[mask]

    weights = weights[mask]

    mean = np.average(values, weights=weights)

    return float(np.average((values - mean) ** 2, weights=weights))



def stabilize_cov(cov, floor=1e-4):

    # Make a covariance matrix numerically stable by flooring tiny eigenvalues

    arr = np.asarray(cov, dtype="float64")

    if arr.ndim == 0:

        arr = np.array([[float(arr)]], dtype="float64")

    arr = np.nan_to_num(arr, nan=0.0)

    arr = 0.5 * (arr + arr.T)

    vals, vecs = np.linalg.eigh(arr)

    vals = np.clip(vals, floor, None)

    return vecs @ np.diag(vals) @ vecs.T



def shrink_corr_to_identity(cov, n_obs, prior_strength=6.0, clip=0.5):

    cov = stabilize_cov(cov, floor=1e-6)

    sd = np.sqrt(np.maximum(np.diag(cov), 1e-8))

    corr = cov / np.outer(sd, sd)

    np.fill_diagonal(corr, 1.0)

    shrink = float(n_obs / (n_obs + prior_strength)) if n_obs > 0 else 0.0

    corr = shrink * corr + (1.0 - shrink) * np.eye(corr.shape[0], dtype="float64")

    corr = np.clip(corr, -clip, clip)

    np.fill_diagonal(corr, 1.0)

    corr = stabilize_cov(corr, floor=1e-6)

    corr_sd = np.sqrt(np.maximum(np.diag(corr), 1e-8))

    corr = corr / np.outer(corr_sd, corr_sd)

    np.fill_diagonal(corr, 1.0)

    return corr



def percent_table(df, cols):

    # Multiply columns by 100 for display as percentages

    out = df.copy()

    for col in cols:

        out[col] = out[col] * 100

    return out



def weighted_composition_average(df, cols, weights, eps=composition_floor):

    comp = close_comp(df, cols, eps=eps).to_numpy(dtype="float64")

    weights = np.asarray(weights, dtype="float64")

    mask = np.isfinite(weights) & (weights > 0)

    if mask.sum() == 0:

        weights = np.ones(comp.shape[0], dtype="float64")

    else:

        comp = comp[mask]

        weights = weights[mask]

    weights = weights / weights.sum()

    mean_share = weights @ comp

    return close_comp(pd.DataFrame([dict(zip(cols, mean_share))]), cols, eps=eps).iloc[0].to_dict()



# Shift district ALR values so their weighted average matches the national target

def align_local_alr_to_national(local_alr, target_alr, weights, cols, baseline="other", max_iter=10, tol=1e-10):

    arr = np.asarray(local_alr, dtype="float64")

    if arr.ndim == 1:

        arr = arr[None, :]

    target = np.asarray(target_alr, dtype="float64").reshape(-1)

    offset = target.copy()

    for _ in range(max_iter):

        adjusted = arr + offset[None, :]

        current_comp = inverse_alr(adjusted, cols, baseline=baseline)

        current_share = weighted_composition_average(current_comp, cols, weights)

        current_alr = alr(pd.DataFrame([current_share]), cols, baseline=baseline).iloc[0].to_numpy(dtype="float64")

        step = target - current_alr

        offset = offset + step

        if np.max(np.abs(step)) < tol:

            break

    return arr + offset[None, :]



def recode_small_categories(train_ser, test_ser, min_count=5):

    train_ser = train_ser.astype("string").fillna("Unknown").str.strip().replace("", "Unknown")

    keep = train_ser.value_counts()[lambda s: s >= min_count].index.tolist()

    if "Other" not in keep:

        keep = keep + ["Other"]

    train_ser = train_ser.where(train_ser.isin(keep), "Other")

    test_ser = test_ser.astype("string").fillna("Unknown").str.strip().replace("", "Unknown")

    test_ser = test_ser.where(test_ser.isin(keep), "Other")

    return train_ser, test_ser, keep



def build_design(train_df, test_df, numeric_cols, cat_specs):

    train_parts = [pd.DataFrame({"intercept": np.ones(len(train_df))}, index=train_df.index)]

    test_parts = [pd.DataFrame({"intercept": np.ones(len(test_df))}, index=test_df.index)]

    meta = {"numeric": {}, "cats": {}}



    for col in numeric_cols:

        mean = float(train_df[col].mean()) if pd.notna(train_df[col].mean()) else 0.0

        std = float(train_df[col].std()) if pd.notna(train_df[col].std()) else 1.0

        if not np.isfinite(std) or std == 0:

            std = 1.0

        train_parts.append(pd.DataFrame({col: (train_df[col].fillna(mean) - mean) / std}, index=train_df.index))

        test_parts.append(pd.DataFrame({col: (test_df[col].fillna(mean) - mean) / std}, index=test_df.index))

        meta["numeric"][col] = {"mean": mean, "std": std}



    for col, min_count in cat_specs:

        train_ser, test_ser, keep = recode_small_categories(train_df[col], test_df[col], min_count=min_count)

        dtrain = pd.get_dummies(train_ser, prefix=col, dtype="float64")

        dtest = pd.get_dummies(test_ser, prefix=col, dtype="float64").reindex(columns=dtrain.columns, fill_value=0.0)

        if dtrain.shape[1] > 1:

            dtrain = dtrain.iloc[:, 1:]

            dtest = dtest[dtrain.columns]

        train_parts.append(dtrain)

        test_parts.append(dtest)

        meta["cats"][col] = {"keep": keep, "columns": dtrain.columns.tolist()}



    X_train = pd.concat(train_parts, axis=1)

    X_test = pd.concat(test_parts, axis=1).reindex(columns=X_train.columns, fill_value=0.0)

    return X_train, X_test, meta



def build_design_from_meta(df, meta):

    parts = [pd.DataFrame({"intercept": np.ones(len(df))}, index=df.index)]

    for col, info in meta["numeric"].items():

        mean = info["mean"]

        std = info["std"]

        parts.append(pd.DataFrame({col: (df[col].fillna(mean) - mean) / std}, index=df.index))

    for col, info in meta["cats"].items():

        ser = df[col].astype("string").fillna("Unknown").str.strip().replace("", "Unknown")

        ser = ser.where(ser.isin(info["keep"]), "Other")

        dtest = pd.get_dummies(ser, prefix=col, dtype="float64").reindex(columns=info["columns"], fill_value=0.0)

        parts.append(dtest)

    return pd.concat(parts, axis=1)



def feature_coverage_table(df, feature_cols):

    rows = []

    for col in feature_cols:

        if col not in df.columns:

            continue

        rows.append(

            {

                "feature": col,

                "nonmissing_share": float(df[col].notna().mean()),

            }

        )

    if not rows:

        return pd.DataFrame(columns=["feature", "nonmissing_share"])

    return pd.DataFrame(rows).sort_values(["nonmissing_share", "feature"]).reset_index(drop=True)



def screen_sparse_features(df, feature_cols, min_nonmissing_share=0.95, grouped_prefixes=None):

    valid_features = [col for col in feature_cols if col in df.columns]

    coverage = {

        col: float(df[col].notna().mean())

        for col in valid_features

    }

    kept = [

        col

        for col in valid_features

        if coverage[col] >= float(min_nonmissing_share)

    ]



    if grouped_prefixes is not None:

        for flag_col, value_prefix in grouped_prefixes:

            if flag_col not in kept:

                continue

            if not any(col.startswith(value_prefix) for col in kept):

                kept.remove(flag_col)



    dropped = [col for col in valid_features if col not in kept]

    coverage_df = pd.DataFrame(

        {

            "feature": valid_features,

            "nonmissing_share": [coverage[col] for col in valid_features],

            "used_downstream": [col in kept for col in valid_features],

        }

    ).sort_values(["used_downstream", "nonmissing_share", "feature"], ascending=[False, False, True]).reset_index(drop=True)

    return kept, dropped, coverage_df



def fit_random_intercept(train_df, target_col, numeric_cols, cat_specs, group_col):

    X_train, _, meta = build_design(train_df, train_df, numeric_cols, cat_specs)

    y_train = train_df[target_col].to_numpy(dtype="float64")

    groups = train_df[group_col].astype("string").fillna("Unknown").to_numpy()

    try:

        model = sm.MixedLM(y_train, X_train.to_numpy(dtype="float64"), groups=groups)

        result = model.fit(reml=False, method="lbfgs", maxiter=400, disp=False)

        rand_map = {}

        for key, value in result.random_effects.items():

            arr = np.asarray(value, dtype="float64").reshape(-1)

            rand_map[str(key)] = float(arr[0]) if len(arr) else 0.0

        return {

            "kind": "mixed",

            "result": result,

            "meta": meta,

            "columns": X_train.columns.tolist(),

            "group_col": group_col,

            "rand_map": rand_map,

            "fallback": float(np.nanmean(y_train)) if np.isfinite(y_train).any() else 0.0,

        }

    except Exception:

        result = sm.OLS(y_train, X_train.to_numpy(dtype="float64")).fit()

        return {

            "kind": "ols",

            "result": result,

            "meta": meta,

            "columns": X_train.columns.tolist(),

            "group_col": group_col,

            "rand_map": {},

            "fallback": float(np.nanmean(y_train)) if np.isfinite(y_train).any() else 0.0,

        }



def predict_random_intercept(bundle, df):

    X = build_design_from_meta(df, bundle["meta"]).reindex(columns=bundle["columns"], fill_value=0.0)

    fixed = X.to_numpy(dtype="float64") @ np.asarray(bundle["result"].params[: X.shape[1]], dtype="float64")

    if bundle["kind"] == "mixed":

        group_effect = (

            df[bundle["group_col"]]

            .astype("string")

            .fillna("Unknown")

            .astype(str)

            .map(bundle["rand_map"])

            .fillna(0.0)

            .to_numpy(dtype="float64")

        )

        pred = fixed + group_effect

    else:

        pred = fixed

    pred = np.asarray(pred, dtype="float64")

    pred = np.where(np.isfinite(pred), pred, bundle["fallback"])

    return pred



def predict_group_effect(bundle, df):

    if bundle["kind"] != "mixed":

        return np.zeros(len(df), dtype="float64")

    return (

        df[bundle["group_col"]]

        .astype("string")

        .fillna("Unknown")

        .astype(str)

        .map(bundle["rand_map"])

        .fillna(0.0)

        .to_numpy(dtype="float64")

    )



def grouped_balanced_folds(df, group_col, n_folds=5):

    counts = (

        df[group_col]

        .astype("string")

        .fillna("Unknown")

        .astype(str)

        .value_counts()

        .sort_values(ascending=False)

    )

    if counts.empty:

        return []

    n_folds = max(1, min(int(n_folds), len(counts)))

    fold_bins = [{"groups": [], "size": 0} for _ in range(n_folds)]

    for group, size in counts.items():

        chosen = min(fold_bins, key=lambda item: item["size"])

        chosen["groups"].append(group)

        chosen["size"] += int(size)

    return [item["groups"] for item in fold_bins if item["groups"]]



def ridge_fit(X, Y, weights, alpha=10.0):

    # Fit weighted ridge regression. Alpha controls how much we shrink coefficients.

    X = np.asarray(X, dtype="float64")

    Y = np.asarray(Y, dtype="float64")

    if Y.ndim == 1:

        Y = Y[:, None]

    weights = np.asarray(weights, dtype="float64")

    weights = np.where(np.isfinite(weights) & (weights > 0), weights, 1.0)

    sw = np.sqrt(weights)[:, None]

    penalty = np.eye(X.shape[1], dtype="float64")

    penalty[0, 0] = 0.0

    beta = np.linalg.solve((X * sw).T @ (X * sw) + alpha * penalty, (X * sw).T @ (Y * sw))

    return beta



def fit_ridge_bundle(train_df, target_cols, numeric_cols, cat_specs, weight_col, alpha=10.0):

    X_train, _, meta = build_design(train_df, train_df, numeric_cols, cat_specs)

    y_train = train_df[target_cols].to_numpy(dtype="float64")

    weights = train_df[weight_col].fillna(1.0).to_numpy(dtype="float64")

    beta = ridge_fit(X_train.to_numpy(dtype="float64"), y_train, weights, alpha=alpha)

    return {

        "kind": "ridge",

        "beta": beta,

        "meta": meta,

        "columns": X_train.columns.tolist(),

        "target_cols": list(target_cols),

        "alpha": float(alpha),

    }



def predict_ridge_bundle(bundle, df):

    X = build_design_from_meta(df, bundle["meta"]).reindex(columns=bundle["columns"], fill_value=0.0)

    return X.to_numpy(dtype="float64") @ bundle["beta"]



def transform_media_per_1000_log(series, pop_total):

    series = pd.to_numeric(series, errors="coerce")

    pop_total = pd.to_numeric(pop_total, errors="coerce")

    return np.log1p(series.mul(1000.0).div(pop_total).replace([np.inf, -np.inf], np.nan))



def build_pro_gov_avg_feature(df, raw_cols):

    transformed = pd.DataFrame(index=df.index)



    for raw_col in raw_cols:

        if raw_col in df.columns:

            transformed[raw_col] = transform_media_per_1000_log(df[raw_col], df["pop_total"])



    if transformed.shape[1] == 0:

        return pd.Series(np.nan, index=df.index, dtype="float64")



    return transformed.mean(axis=1)



def build_expanded_structure_features(source_df):

    # Turn raw district statistics into per-1,000-resident rates, shares, and log transforms

    df = source_df.copy()



    raw_numeric_inputs = [

        "pop_total",

        "pop_0_14",

        "pop_65_plus",

        "pop_perm",

        "pop_perm_0_14_female",

        "pop_perm_65_plus_female",

        "pop_perm_0_14_male",

        "pop_perm_65_plus_male",

        "pop_perm_0_24",

        "budget_revenue",

        "budget_expenditure",

        "dwellings_built",

        "dwellings_terminated",

        "dwellings_built_muni",

        "dwellings_built_private",

        "gps_doctors",

        "pediatric_doctors",

        "cars",

        "internet_other",

        "tv_subs",

        "cars_first_reg",

        "post_offices",

        "internet_total",

        "internet_xdsl",

        "internet_cable",

        "internet_optical",

        "unemployed_total",

        "unemployed_180_plus",

        "unemployed_univ",

        "unemployed_bluecollar",

        "unemployed_young",

        "unemployed_365_plus",

        "labour_income_million_huf_t_3",

        "labor_income_million_huf_t_3_real2010",

        "cpi",

    ]



    for col in raw_numeric_inputs:

        if col in df.columns:

            df[col] = pd.to_numeric(df[col], errors="coerce")



    pop_total = pd.to_numeric(df["pop_total"], errors="coerce")

    pop_denom = pop_total.replace({0: np.nan})

    rate_scale = 1000.0

    feature_rows = []



    def add_feature(feature_name, values, source_col, transform):

        df[feature_name] = values

        feature_rows.append(

            {

                "feature": feature_name,

                "source_col": source_col,

                "transform": transform,

            }

        )



    add_feature("log_pop_total", np.log1p(pop_total), "pop_total", "log")



    share_features = {

        "pop_0_14": "young_share",

        "pop_65_plus": "old_share",

        "pop_perm": "perm_population_share",

        "pop_perm_0_14_female": "perm_0_14_female_share",

        "pop_perm_65_plus_female": "perm_65_plus_female_share",

        "pop_perm_0_14_male": "perm_0_14_male_share",

        "pop_perm_65_plus_male": "perm_65_plus_male_share",

        "pop_perm_0_24": "perm_0_24_share",

    }



    for source_col, feature_name in share_features.items():

        if source_col in df.columns:

            add_feature(feature_name, df[source_col] / pop_denom, source_col, "share_of_pop_total")



    per_1000_features = {

        "budget_revenue": "budget_revenue_per_capita",

        "budget_expenditure": "budget_expenditure_per_capita",

        "dwellings_built": "dwellings_per_capita",

        "dwellings_terminated": "dwellings_terminated_per_capita",

        "dwellings_built_muni": "dwellings_built_muni_per_capita",

        "dwellings_built_private": "dwellings_built_private_per_capita",

        "gps_doctors": "gps_doctors_per_capita",

        "pediatric_doctors": "pediatric_doctors_per_capita",

        "cars": "cars_per_capita",

        "internet_other": "internet_other_per_capita",

        "tv_subs": "tv_subs_per_capita",

        "cars_first_reg": "cars_first_reg_per_capita",

        "post_offices": "post_offices_per_capita",

        "internet_total": "internet_per_capita",

        "internet_xdsl": "internet_xdsl_per_capita",

        "internet_cable": "internet_cable_per_capita",

        "internet_optical": "internet_optical_per_capita",

        "unemployed_total": "unemployed_per_capita",

        "unemployed_180_plus": "unemployed_180_plus_per_capita",

        "unemployed_univ": "unemployed_univ_per_capita",

        "unemployed_bluecollar": "unemployed_bluecollar_per_capita",

        "unemployed_young": "unemployed_young_per_capita",

        "unemployed_365_plus": "unemployed_365_plus_per_capita",

        "labour_income_million_huf_t_3": "nominal_income_per_capita",

        "labor_income_million_huf_t_3_real2010": "real_income_per_capita",

    }



    for source_col, feature_name in per_1000_features.items():

        if source_col in df.columns:

            add_feature(feature_name, df[source_col] * rate_scale / pop_denom, source_col, "per_1000_residents")



    if {"gps_doctors", "pediatric_doctors"}.issubset(df.columns):

        add_feature(

            "doctor_per_capita",

            rate_scale * (df["gps_doctors"] + df["pediatric_doctors"]) / pop_denom,

            "gps_doctors+pediatric_doctors",

            "per_1000_residents_sum",

        )



    if {"budget_revenue", "budget_expenditure"}.issubset(df.columns):

        add_feature(

            "budget_balance_per_capita",

            rate_scale * (df["budget_revenue"] - df["budget_expenditure"]) / pop_denom,

            "budget_revenue-budget_expenditure",

            "per_1000_residents_difference",

        )



    if "cpi" in df.columns:

        add_feature("cpi", df["cpi"], "cpi", "raw")



    feature_metadata = pd.DataFrame(feature_rows)

    structure_feature_candidates = feature_metadata["feature"].tolist()

    return df, structure_feature_candidates, feature_metadata




def build_condition_numbers(X, weights, alpha_grid):
    """Condition number of the ridge normal equations for each alpha in the grid.

    We compute cond((X^T W X + alpha * P)) where P is the ridge penalty matrix
    (identity with a zero in the intercept position). A smaller condition number
    means the linear system is more numerically stable.
    """
    X = np.asarray(X, dtype="float64")
    weights = np.asarray(weights, dtype="float64")
    weights = np.where(np.isfinite(weights) & (weights > 0), weights, 1.0)
    sw = np.sqrt(weights)[:, None]
    XtX = (X * sw).T @ (X * sw)
    penalty = np.eye(X.shape[1], dtype="float64")
    penalty[0, 0] = 0.0  # intercept is unpenalized
    return [float(np.linalg.cond(XtX + float(a) * penalty)) for a in alpha_grid]


def select_alpha_mae_cond(alpha_df, mae_col, cond_col, mae_tol, cond_threshold):
    """Select alpha: lowest MAE is primary; condition number is the tiebreaker.

    Among all alphas within mae_tol of the best MAE, prefer those whose
    condition number stays below cond_threshold. If none qualify, fall back
    to the lowest-MAE candidate regardless of condition number.
    """
    best_mae = float(alpha_df[mae_col].min())
    close = alpha_df[alpha_df[mae_col] <= best_mae + mae_tol].copy()
    well_cond = close[close[cond_col] < cond_threshold]
    pool = well_cond if not well_cond.empty else close
    return float(pool.sort_values([mae_col, cond_col]).iloc[0]["alpha"])


print("Helper functions defined.")

Helper functions defined.


Interpretation. Lower coverage means the variable is harder to use safely without imputation.


## Data Construction

Load screened feature config if available.


In [8]:
# Load screened feature config if available

screened_feature_path = project_root / "data" / "created" / "screened_feature_sets.json"

if screened_feature_path.exists():

    screened_feature_config = json.loads(screened_feature_path.read_text(encoding="utf-8"))

downstream_recent_media_config = screened_feature_config.get("downstream_recent_media_config", {})
if isinstance(downstream_recent_media_config, dict):
    config_outlets = downstream_recent_media_config.get("selected_media_outlets")
    config_window = downstream_recent_media_config.get("selected_media_window")
    config_metric = downstream_recent_media_config.get("selected_media_metric")
    config_feature_cols = downstream_recent_media_config.get("selected_media_feature_cols")
    config_recent_features = downstream_recent_media_config.get("selected_recent_media_features")
    config_raw_cols = downstream_recent_media_config.get("selected_media_raw_cols")
    if (
        config_feature_cols == ["pro_gov_avg"]
        and config_recent_features == ["pro_gov_avg"]
        and isinstance(config_window, str)
        and isinstance(config_metric, str)
        and isinstance(config_outlets, list)
        and config_outlets
        and all(isinstance(outlet, str) for outlet in config_outlets)
    ):
        selected_single_media_outlet = downstream_recent_media_config.get(
            "selected_single_media_outlet",
            selected_single_media_outlet,
        )
        selected_media_window = config_window
        selected_media_metric = config_metric
        selected_media_outlets = config_outlets.copy()
        selected_media_raw_cols = [
            f"{selected_media_metric}_{selected_media_window}_{outlet}"
            for outlet in selected_media_outlets
        ]
        if (
            isinstance(config_raw_cols, list)
            and len(config_raw_cols) == len(selected_media_raw_cols)
            and all(isinstance(raw_col, str) for raw_col in config_raw_cols)
        ):
            selected_media_raw_cols = config_raw_cols.copy()



selected_recent_media_features = selected_media_feature_cols.copy()

screened_full_history_structure = [

    feature

    for feature in screened_feature_config.get("screened_full_history_structure", screened_full_history_structure)

]

preliminary_fundamentals_models = screened_feature_config.get(

    "preliminary_fundamentals_models",

    preliminary_fundamentals_models,

)



# Build block vote aggregates from district results

block_votes = final_results_oevk_party.groupby(

    ["year", "election_type", "oevk_id", "party_block"],

    as_index=False,

)["list_votes"].sum()



district_totals = final_results_oevk_party.groupby(

    ["year", "election_type", "oevk_id"],

    as_index=False,

).agg(

    valid_list_total=("valid_list_total", "first"),

    valid_cand_total=("valid_cand_total", "first"),

    electorate_list=("electorate_list", "first"),

)



actual_national = (

    block_votes.merge(district_totals, on=["year", "election_type", "oevk_id"], how="left")

    .groupby(["year", "election_type", "party_block"], as_index=False)

    .agg(list_votes=("list_votes", "sum"), valid_total=("valid_list_total", "sum"))

)

actual_national["share"] = actual_national["list_votes"] / actual_national["valid_total"]

actual_national["election"] = actual_national["year"].astype(str) + "_" + actual_national["election_type"]



actual_national_wide = actual_national.pivot(index="election", columns="party_block", values="share").fillna(0.0)

for block in block_cols:

    if block not in actual_national_wide.columns:

        actual_national_wide[block] = 0.0

actual_national_wide = actual_national_wide[block_cols].reset_index()



actual_national_alr = alr(actual_national_wide[block_cols], block_cols)

actual_national_alr.columns = [f"actual_alr_{block}" for block in nonbase_blocks]

actual_national_wide = pd.concat([actual_national_wide[["election"]], actual_national_wide[block_cols], actual_national_alr], axis=1)



display(percent_table(actual_national_wide[["election"] + block_cols], block_cols).round(2))



# Poll block maps

poll_block_map_default = {
    "Fidesz": "gov",
    "MSZP": "other",
    "LMP": "other",
    "DK": "other",
    "Együtt": "other",
    "P": "other",
    "MM": "other",
    "DK-MSZP-P": "other",
    "MSZP-P": "other",
    "MMN": "other",
    "NP": "other",
    "EM": "other",
    "MKKP": "other",
    "2RK": "other",
    "TISZA": "other",
    "Jobbik": "other",
    "MH": "opp_radical",
    "Egyéb párt": "other",
}

poll_block_map_2018 = poll_block_map_default.copy()
for party in ["MSZP", "P", "MSZP-P", "DK", "LMP", "MM"]:
    poll_block_map_2018[party] = "opp"
poll_block_map_2018["Jobbik"] = "opp_radical"
poll_block_map_2018["MH"] = "other"

poll_block_map_2019_ep = poll_block_map_default.copy()
for party in ["MSZP", "P", "MSZP-P", "DK", "LMP", "MM", "Jobbik"]:
    poll_block_map_2019_ep[party] = "opp"

poll_block_map_2022_nat = poll_block_map_default.copy()
for party in ["MSZP", "P", "MSZP-P", "DK", "LMP", "MM", "EM", "Jobbik"]:
    poll_block_map_2022_nat[party] = "opp"

poll_block_map_2024_ep = poll_block_map_default.copy()
for party in ["TISZA", "DK-MSZP-P"]:
    poll_block_map_2024_ep[party] = "opp"

poll_block_map_2026_nat = poll_block_map_default.copy()
poll_block_map_2026_nat["TISZA"] = "opp"

all_poll_party_cols = sorted(
    {
        col
        for mapping in [
            poll_block_map_default,
            poll_block_map_2018,
            poll_block_map_2019_ep,
            poll_block_map_2022_nat,
            poll_block_map_2024_ep,
            poll_block_map_2026_nat,
        ]
        for col in mapping
    }
)

for col in all_poll_party_cols:
    if col in polls.columns:
        polls[col] = clean_num(polls[col]).fillna(0.0)

polls["sample_size"] = clean_num(polls["Minta"])
polls["basis"] = polls["Adatok.bázisa"].astype("string").str.strip()
polls["focus"] = polls["Fókusz"].astype("string").str.strip()
polls["mode"] = polls["Mód"].astype("string").str.strip().replace("", "Unknown").fillna("Unknown")
polls["pollster"] = polls["Adatgazda"].astype("string").str.strip().replace("", "Unknown").fillna("Unknown")
polls = polls[polls["basis"].isin(["Összes", "Biztos"])].copy()

election_meta = {
    "2018_nat": {"date": pd.Timestamp("2018-04-08"), "type": "nat", "focus": "nat"},
    "2019_ep": {"date": pd.Timestamp("2019-05-26"), "type": "ep", "focus": "EP"},
    "2022_nat": {"date": pd.Timestamp("2022-04-03"), "type": "nat", "focus": "nat"},
    "2024_ep": {"date": pd.Timestamp("2024-06-09"), "type": "ep", "focus": "EP"},
    "2026_nat": {"date": election_assumption_2026, "type": "nat", "focus": "nat"},
}

poll_block_map_by_election = {
    "2018_nat": poll_block_map_2018,
    "2019_ep": poll_block_map_2019_ep,
    "2022_nat": poll_block_map_2022_nat,
    "2024_ep": poll_block_map_2024_ep,
    "2026_nat": poll_block_map_2026_nat,
}

poll_rows = []

for election, meta in election_meta.items():

    if meta["focus"] == "EP":

        focus_mask = polls["focus"].fillna("").eq("EP")

    else:

        focus_mask = polls["focus"].fillna("").eq("")



    mask = (

        focus_mask

        & polls["Vég"].notna()

        & (polls["Vég"] <= meta["date"])

        & (polls["Vég"] >= meta["date"] - pd.Timedelta(days=365))

        & polls["sample_size"].notna()

        & (polls["sample_size"] > 0)

    )



    sub = polls.loc[mask].copy()

    if sub.empty:

        continue



    active_map = poll_block_map_by_election.get(election, poll_block_map_default)

    for block in block_cols:

        party_cols = [col for col, mapped in active_map.items() if mapped == block and col in sub.columns]

        sub[block] = sub[party_cols].sum(axis=1) if party_cols else 0.0



    comp = close_comp(sub[block_cols].div(100.0), block_cols)

    obs_alr = alr(comp, block_cols)

    sub = pd.concat([sub.reset_index(drop=True), obs_alr.reset_index(drop=True)], axis=1)

    sub["undecided"] = clean_num(sub["Egyéb válasz"]).fillna(0.0) / 100.0

    sub["is_biztos"] = (sub["basis"] == "Biztos").astype(float)

    sub["days_to_election"] = (meta["date"] - sub["Vég"]).dt.days.astype("float64")

    sub["election"] = election

    sub["election_type"] = meta["type"]



    keep_cols = [

        "election",

        "election_type",

        "Vég",

        "pollster",

        "mode",

        "basis",

        "sample_size",

        "undecided",

        "is_biztos",

        "days_to_election",

    ] + [f"alr_{block}" for block in nonbase_blocks]

    poll_rows.append(sub[keep_cols])



poll_df = pd.concat(poll_rows, ignore_index=True)

simple_poll_window_days = 60

# Merge actual results onto poll_df

poll_df = poll_df.merge(actual_national_wide, on="election", how="left")



# Build district result wide table

result_wide = (

    final_results_oevk_party.groupby(

        ["year", "election_type", "oevk_id", "party_block"],

        as_index=False,

    )[["list_share", "cand_share", "list_votes", "cand_votes"]].sum()

    .pivot(index=["year", "election_type", "oevk_id"], columns="party_block", values=["list_share", "cand_share", "list_votes", "cand_votes"])

)

result_wide.columns = [f"{left}_{right}" for left, right in result_wide.columns]

result_wide = result_wide.reset_index()



for metric in ["list_share", "cand_share"]:

    for block in block_cols:

        col = f"{metric}_{block}"

        if col not in result_wide.columns:

            result_wide[col] = 0.0



result_wide = result_wide.merge(district_totals, on=["year", "election_type", "oevk_id"], how="left")



# Build stat features

stat_features, structure_feature_candidates, structure_feature_metadata = build_expanded_structure_features(final_statistics_oevk)

stat_features["pro_gov_avg"] = build_pro_gov_avg_feature(stat_features, selected_media_raw_cols)



feature_keep = [

    "year",

    "oevk_id",

    "county",

    "stat_region",

    "pop_total",

] + [

    feature for feature in screened_full_history_structure if feature in stat_features.columns

] + [

    feature for feature in selected_media_feature_cols if feature in stat_features.columns

]

feature_keep = list(dict.fromkeys(feature_keep))

stat_features = stat_features[feature_keep].copy()



# Previous election result lookup tables

nat_lookup = result_wide[result_wide["election_type"] == "nat"][["year", "oevk_id"] + [f"list_share_{block}" for block in block_cols] + ["valid_list_total", "valid_cand_total", "electorate_list"]].copy()

ep_lookup = result_wide[result_wide["election_type"] == "ep"][["year", "oevk_id"] + [f"list_share_{block}" for block in block_cols]].copy()



nat_prev_map = {2019: 2018, 2022: 2018, 2024: 2022, 2026: 2022}

ep_prev_map = {2022: 2019, 2024: 2019, 2026: 2024}



def add_share_lags(df):

    out = df.copy()

    for prefix in ["lag_nat", "lag_ep"]:

        for block in block_cols:

            out[f"{prefix}_{block}"] = np.nan

    out["lag_nat_valid_list_total"] = np.nan

    out["lag_nat_valid_cand_total"] = np.nan

    out["lag_nat_electorate"] = np.nan



    for target_year, source_year in nat_prev_map.items():

        rename_map = {f"list_share_{block}": f"lag_nat_{block}" for block in block_cols}

        rename_map["valid_list_total"] = "lag_nat_valid_list_total"

        rename_map["valid_cand_total"] = "lag_nat_valid_cand_total"

        rename_map["electorate_list"] = "lag_nat_electorate"

        src = nat_lookup[nat_lookup["year"] == source_year].rename(columns=rename_map).drop(columns="year")

        mask = out["year"] == target_year

        merged = out.loc[mask, ["oevk_id"]].merge(src, on="oevk_id", how="left")

        for block in block_cols:

            out.loc[mask, f"lag_nat_{block}"] = merged[f"lag_nat_{block}"].to_numpy()

        out.loc[mask, "lag_nat_valid_list_total"] = merged["lag_nat_valid_list_total"].to_numpy()

        out.loc[mask, "lag_nat_valid_cand_total"] = merged["lag_nat_valid_cand_total"].to_numpy()

        out.loc[mask, "lag_nat_electorate"] = merged["lag_nat_electorate"].to_numpy()



    for target_year, source_year in ep_prev_map.items():

        rename_map = {f"list_share_{block}": f"lag_ep_{block}" for block in block_cols}

        src = ep_lookup[ep_lookup["year"] == source_year].rename(columns=rename_map).drop(columns="year")

        mask = out["year"] == target_year

        merged = out.loc[mask, ["oevk_id"]].merge(src, on="oevk_id", how="left")

        for block in block_cols:

            out.loc[mask, f"lag_ep_{block}"] = merged[f"lag_ep_{block}"].to_numpy()



    out["has_lag_nat"] = out[[f"lag_nat_{block}" for block in block_cols]].notna().all(axis=1).astype(float)

    out["has_lag_ep"] = out[[f"lag_ep_{block}" for block in block_cols]].notna().all(axis=1).astype(float)

    out["forecast_weight"] = out["lag_nat_valid_list_total"].fillna(out["lag_nat_electorate"]).fillna(out["pop_total"]).fillna(1.0)

    return out



# Build full prior_base

prior_base = add_share_lags(result_wide.merge(stat_features, on=["year", "oevk_id"], how="left"))

prior_base = prior_base.merge(

    alr(

        prior_base[[f"list_share_{block}" for block in block_cols]].rename(columns={f"list_share_{block}": block for block in block_cols}),

        block_cols,

    ),

    left_index=True,

    right_index=True,

    how="left",

)

prior_base = prior_base.rename(columns={f"alr_{block}": f"target_alr_{block}" for block in nonbase_blocks})

prior_base["fit_weight"] = prior_base["valid_list_total"].fillna(prior_base["electorate_list"]).fillna(1.0)

prior_base["election"] = prior_base["year"].astype(str) + "_" + prior_base["election_type"]



# Build lean hist

lean_hist = result_wide.merge(stat_features, on=["year", "oevk_id"], how="left")

district_list_input = lean_hist[[f"list_share_{block}" for block in block_cols]].rename(columns={f"list_share_{block}": block for block in block_cols})

district_list_alr = alr(district_list_input, block_cols)

district_list_alr.columns = [f"list_alr_{block}" for block in nonbase_blocks]

lean_hist = pd.concat([lean_hist.reset_index(drop=True), district_list_alr.reset_index(drop=True)], axis=1)



national_rows = []

for (year, etype), group in lean_hist.groupby(["year", "election_type"]):

    weights = group["valid_list_total"].fillna(1.0).to_numpy(dtype="float64")

    row = {"year": year, "election_type": etype}

    nat_share = {}

    for block in block_cols:

        nat_share[block] = safe_weighted_average(group[f"list_share_{block}"], weights)

    nat_alr = alr(pd.DataFrame([nat_share]), block_cols).iloc[0]

    for block in block_cols:

        row[f"nat_{block}"] = nat_share[block]

    for block in nonbase_blocks:

        row[f"nat_alr_{block}"] = float(nat_alr[f"alr_{block}"])

    national_rows.append(row)



national_level = pd.DataFrame(national_rows)

lean_hist = lean_hist.merge(national_level, on=["year", "election_type"], how="left")

for block in nonbase_blocks:

    lean_hist[f"lean_alr_{block}"] = lean_hist[f"list_alr_{block}"] - lean_hist[f"nat_alr_{block}"]

lean_hist["election"] = lean_hist["year"].astype(str) + "_" + lean_hist["election_type"]



# Lean lag lookups

nat_lean_lookup = lean_hist[lean_hist["election_type"] == "nat"][["year", "oevk_id"] + [f"lean_alr_{block}" for block in nonbase_blocks]].copy()

ep_lean_lookup = lean_hist[lean_hist["election_type"] == "ep"][["year", "oevk_id"] + [f"lean_alr_{block}" for block in nonbase_blocks]].copy()



def add_lean_lags(df):

    out = df.copy()

    for prefix in ["lag_nat_lean", "lag_ep_lean"]:

        for block in nonbase_blocks:

            out[f"{prefix}_{block}"] = np.nan



    for target_year, source_year in nat_prev_map.items():

        rename_map = {f"lean_alr_{block}": f"lag_nat_lean_{block}" for block in nonbase_blocks}

        src = nat_lean_lookup[nat_lean_lookup["year"] == source_year].rename(columns=rename_map).drop(columns="year")

        mask = out["year"] == target_year

        merged = out.loc[mask, ["oevk_id"]].merge(src, on="oevk_id", how="left")

        for block in nonbase_blocks:

            out.loc[mask, f"lag_nat_lean_{block}"] = merged[f"lag_nat_lean_{block}"].to_numpy()



    for target_year, source_year in ep_prev_map.items():

        rename_map = {f"lean_alr_{block}": f"lag_ep_lean_{block}" for block in nonbase_blocks}

        src = ep_lean_lookup[ep_lean_lookup["year"] == source_year].rename(columns=rename_map).drop(columns="year")

        mask = out["year"] == target_year

        merged = out.loc[mask, ["oevk_id"]].merge(src, on="oevk_id", how="left")

        for block in nonbase_blocks:

            out.loc[mask, f"lag_ep_lean_{block}"] = merged[f"lag_ep_lean_{block}"].to_numpy()



    out["has_lag_nat_lean"] = out[[f"lag_nat_lean_{block}" for block in nonbase_blocks]].notna().all(axis=1).astype(float)

    out["has_lag_ep_lean"] = out[[f"lag_ep_lean_{block}" for block in nonbase_blocks]].notna().all(axis=1).astype(float)

    return out



structure_cat_specs = [("stat_region", 1), ("election_type", 1)]

prior_cat_specs = structure_cat_specs

share_lag_features = ["has_lag_nat", "has_lag_ep"] + [f"lag_nat_{block}" for block in block_cols] + [f"lag_ep_{block}" for block in block_cols]

recent_correction_cat_specs = [("election_type", 1)]



print("Data construction complete.")

print("poll_df shape:", poll_df.shape)

print("prior_base shape:", prior_base.shape)

print("lean_hist shape:", lean_hist.shape)


,election,gov,opp,opp_radical,other
0,2018_nat,47.33,28.49,19.77,4.41
1,2019_ep,51.83,41.76,3.34,3.07
2,2022_nat,52.10,35.99,6.11,5.79
3,2024_ep,44.28,38.03,6.78,10.91


Data construction complete.
poll_df shape: (507, 20)
prior_base shape: (424, 55)
lean_hist shape: (424, 50)


Interpretation. This output is a quick diagnostic check of scale, spread and completeness.


## Poll Layer Helper Functions

This block defines helper functions that later cells reuse.


In [9]:
# this block defines helper functions that later cells reuse.
def aggregate_poll_rows_to_share(pred_df, share_cols, weight_col):

    pred_map = {}

    for block in block_cols:

        pred_map[block] = safe_weighted_average(pred_df[share_cols[block]], pred_df[weight_col])

    return close_comp(pd.DataFrame([pred_map]), block_cols, eps=composition_floor).iloc[0].to_dict()



def summarize_simple_polls(poll_df_sub, election_name, window_days=60, actual_wide=None):

    sub = poll_df_sub[poll_df_sub["election"] == election_name].copy()

    sub = sub[sub["days_to_election"] <= window_days].copy()

    if sub.empty:

        return pd.DataFrame()

    comp = inverse_alr(sub[[f"alr_{block}" for block in nonbase_blocks]].to_numpy(dtype="float64"), block_cols)

    rows = []

    for block in block_cols:

        shares = comp[block].to_numpy(dtype="float64")

        weights = sub["sample_size"].to_numpy(dtype="float64")

        mean = np.average(shares, weights=weights)

        sd = float(np.std(shares, ddof=1)) if len(shares) > 1 else np.nan

        se = float(sd / np.sqrt(len(shares))) if len(shares) > 1 else np.nan

        if actual_wide is not None:

            actual_rows = actual_wide[actual_wide["election"] == election_name]

            if not actual_rows.empty:

                actual_val = float(actual_rows.iloc[0][block])

            else:

                actual_val = np.nan

        else:

            actual_val = np.nan

        rows.append(

            {

                "election": election_name,

                "block": block,

                "mean": mean,

                "sd_between_polls": sd,

                "se": se,

                "actual": actual_val,

                "abs_err": abs(mean - actual_val) if np.isfinite(actual_val) else np.nan,

                "n_polls": len(shares),

            }

        )

    return pd.DataFrame(rows)



def build_bias_floor_map(summary_df, exclude_election=None, fallback=None):

    work = summary_df.copy()

    if exclude_election is not None:

        work = work[work["election"] != exclude_election].copy()

    bias_floor = work.groupby("block")["signed_error"].std(ddof=1).reindex(block_cols)

    if fallback is not None:

        bias_floor = bias_floor.fillna(fallback.reindex(block_cols).fillna(0.0))

    return bias_floor.fillna(0.0)



def summarize_poll_candidate(election, pred_share_map, actual_wide):

    actual_rows = actual_wide[actual_wide["election"] == election]

    if actual_rows.empty:

        return {"election": election, "mae": np.nan, "rmse": np.nan}

    actual_row = actual_rows.iloc[0]

    row = {"election": election}

    sq_errors = []

    for block in block_cols:

        row[f"pred_{block}"] = float(pred_share_map[block])

        row[f"actual_{block}"] = float(actual_row[block])

        row[f"abs_err_{block}"] = abs(row[f"pred_{block}"] - row[f"actual_{block}"])

        sq_errors.append((row[f"pred_{block}"] - row[f"actual_{block}"]) ** 2)

    pred_alr = alr(pd.DataFrame([pred_share_map]), block_cols).iloc[0]

    for block in nonbase_blocks:

        row[f"pred_alr_{block}"] = float(pred_alr[f"alr_{block}"])

        row[f"actual_alr_{block}"] = float(actual_row[f"actual_alr_{block}"])

        row[f"alr_err_{block}"] = row[f"pred_alr_{block}"] - row[f"actual_alr_{block}"]

    row["mae"] = float(np.mean([row[f"abs_err_{block}"] for block in block_cols]))

    row["rmse"] = float(np.sqrt(np.mean(sq_errors)))

    return row



def choose_strict_poll_model(summary_df, mae_tol=5e-4):

    complexity_rank = {

        "simple_weighted_average": 0,

        "simple_average_plus_bias_floor": 1,

        "ridge_poll_model_05": 2,

        "mixed_house_effect_only": 3,

        "house_effects_rw": 4,
        "mixed_plus_random_walk_06": 4,

    }

    ranked = summary_df.copy()

    best_mae = float(ranked["mean_mae"].min())

    close = ranked[ranked["mean_mae"] <= best_mae + mae_tol].copy()

    if "avg_coverage_95" in close.columns and close["avg_coverage_95"].notna().any():

        close["coverage_gap_to_95"] = close["avg_coverage_95"].apply(

            lambda x: abs(float(x) - 0.95) if pd.notna(x) else np.inf

        )

        best_gap = float(close["coverage_gap_to_95"].min())

        close = close[close["coverage_gap_to_95"] <= best_gap + 1e-12].copy()

    close["complexity_rank"] = close["model"].map(complexity_rank).fillna(99).astype(int)

    close = close.sort_values(["complexity_rank", "model"]).reset_index(drop=True)

    return str(close.iloc[0]["model"])




def fit_house_effects(train_df, nonbase_blocks, numeric_features, cat_specs, weight_col):
    """
    Estimate pollster house effects via a mixed-effects model (equation 9 in methodology):
        e_ib = x_i^T * gamma_b + u_c(i)b + eps_ib,   u_c(i)b ~ N(0, sigma_ub^2)
    where e_ib = poll_alr - actual_alr, x_i are fixed covariates (days, mode, etc.),
    and u_c(i)b is the pollster random effect estimated via REML.
    The BLUP shrinkage is automatic: pollsters with few polls are pulled toward zero
    based on the estimated variance ratio sigma_ub^2 / (sigma_ub^2 + sigma_eb^2/n_c).
    No manual prior strength constant is needed.
    Returns {block: {pollster_name: house_effect_in_alr}}.
    """
    from statsmodels.regression.mixed_linear_model import MixedLM

    house_effects = {}
    for block in nonbase_blocks:
        error = (train_df[f"alr_{block}"] - train_df[f"actual_alr_{block}"]).to_numpy(dtype="float64")

        # Build fixed-effects design matrix (no pollster column)
        X_fixed, _, _ = build_design(train_df, train_df, numeric_features, cat_specs)
        X_fixed_arr = X_fixed.to_numpy(dtype="float64")
        groups = train_df["pollster"].astype(str).to_numpy()

        # Weighted MixedLM: pre-scale endog and exog by sqrt(w_i / mean_w) for WLS.
        # statsmodels MixedLM has no native observation-weight parameter, so this
        # pre-multiplication is the standard approach to weight polls by sample size.
        weights_arr = train_df[weight_col].fillna(1.0).to_numpy(dtype="float64")
        weights_arr = np.where(weights_arr > 0, weights_arr, 1.0)
        mean_w = float(weights_arr.mean()) or 1.0
        sqrt_w = np.sqrt(weights_arr / mean_w)

        try:
            model = MixedLM(endog=error * sqrt_w, exog=X_fixed_arr * sqrt_w[:, None], groups=groups)
            result = model.fit(reml=True, disp=False)
            # Extract BLUPs: {pollster: intercept_random_effect}
            effects = {
                str(p): float(v.iloc[0])
                for p, v in result.random_effects.items()
            }
        except Exception:
            # Fallback: simple per-pollster weighted mean (no shrinkage)
            # This can happen when MixedLM fails to converge with very few groups
            effects = {}
            for p in np.unique(groups):
                mask = groups == p
                w_p = train_df[weight_col].fillna(1.0).to_numpy(dtype="float64")[mask]
                r_p = error[mask]
                effects[str(p)] = float(np.average(r_p, weights=w_p))

        house_effects[block] = effects
    return house_effects


def apply_house_effects(poll_df, house_effects, nonbase_blocks):
    """
    Subtract per-pollster house effect from each poll's ALR value.
    Adds alr_corr_{block} columns. Unknown pollsters get zero correction.
    """
    df = poll_df.copy()
    pollsters = df["pollster"].astype(str)
    for block in nonbase_blocks:
        effects = house_effects.get(block, {})
        correction = pollsters.map(effects).fillna(0.0).to_numpy(dtype="float64")
        df[f"alr_corr_{block}"] = df[f"alr_{block}"].to_numpy(dtype="float64") - correction
    return df


def run_random_walk_filter(poll_df, nonbase_blocks, weight_col,
                            days_col="days_to_election",
                            penalty_const=0.01, max_days=365):
    """
    Kalman random-walk filter on (house-effect-corrected) polls.
    1. Aggregate polls to a daily weighted mean and measurement variance
       R_t = within-day_var + penalty_const * (median_N / N_t).
    2. Estimate evolution variance q from day-to-day changes minus 2*mean_R.
    3. Run Kalman filter from farthest day to closest day.
    Returns {block: filtered_ALR_at_last_available_day}.
    """
    result = {}
    for block in nonbase_blocks:
        val_col = f"alr_corr_{block}" if f"alr_corr_{block}" in poll_df.columns else f"alr_{block}"
        df = poll_df[[days_col, val_col, weight_col]].copy()
        df = df[(df[days_col] >= 0) & (df[days_col] <= max_days) & df[val_col].notna()].copy()
        df[weight_col] = df[weight_col].fillna(1.0).clip(lower=0.001)
        if df.empty:
            result[block] = np.nan
            continue

        # Aggregate to daily observations
        daily_rows = []
        for day, grp in df.groupby(days_col):
            w = grp[weight_col].to_numpy(dtype="float64")
            v = grp[val_col].to_numpy(dtype="float64")
            z_t = float(np.average(v, weights=w))
            n_t = float(w.sum())
            var_within = float(np.average((v - z_t) ** 2, weights=w)) if len(v) > 1 else 0.0
            daily_rows.append({"day": int(day), "z": z_t, "n": n_t, "var_within": var_within})
        if not daily_rows:
            result[block] = np.nan
            continue

        # Sort high days first (far from election) to low days (near election)
        daily_df = pd.DataFrame(daily_rows).sort_values("day", ascending=False).reset_index(drop=True)
        n_median = float(daily_df["n"].median())
        daily_df["R"] = daily_df["var_within"] + penalty_const * (n_median / daily_df["n"].clip(lower=0.001))

        # Estimate per-day evolution variance q.
        # For a random walk with a k-day gap: Var(z_{t+k} - z_t) = k*q + 2*R
        # Observations are spaced by varying gaps, so normalize by mean gap:
        #   q = (Var(diffs) - 2*mean_R) / mean_gap
        z_series = daily_df["z"].to_numpy()
        days_arr = daily_df["day"].to_numpy()  # sorted high to low
        step_sizes = np.abs(np.diff(days_arr)).clip(min=1)  # days between consecutive obs
        if len(z_series) > 2:
            diffs = np.diff(z_series)
            var_diffs = float(np.var(diffs, ddof=1))
            mean_steps = float(np.mean(step_sizes))
            q = max((var_diffs - 2.0 * float(daily_df["R"].mean())) / mean_steps, 1e-6)
        else:
            q = 1e-4

        # Kalman filter: predict + update for each observed day
        m = float(daily_df.iloc[0]["z"])
        P = float(daily_df.iloc[0]["R"])
        prev_day = int(daily_df.iloc[0]["day"])

        for _, row in daily_df.iloc[1:].iterrows():
            curr_day = int(row["day"])
            steps = max(prev_day - curr_day, 1)
            P_pred = P + steps * q
            R_t = float(row["R"])
            K = P_pred / (P_pred + R_t)
            m = m + K * (float(row["z"]) - m)
            P = (1.0 - K) * P_pred
            prev_day = curr_day

        result[block] = float(m)
    return result


def allocate_dhondt(votes, seats, threshold=0.05):

    total_votes = sum(votes.values())

    eligible = {k: v for k, v in votes.items() if total_votes > 0 and (v / total_votes) >= threshold}

    if not eligible:

        return {k: 0 for k in votes}

    alloc = {k: 0 for k in eligible}

    for _ in range(seats):

        winner = max(eligible, key=lambda k: eligible[k] / (alloc[k] + 1))

        alloc[winner] += 1

    return {k: alloc.get(k, 0) for k in votes}



def legal_parliament_seats(district_list_votes, district_cand_votes, list_seats=93):

    smd_counts = {block: 0 for block in block_cols}

    compensation = {block: 0.0 for block in block_cols}

    national_list_votes = district_list_votes.sum(axis=0).to_dict()



    for row_idx in range(district_cand_votes.shape[0]):

        row = district_cand_votes.iloc[row_idx]

        ranking = row.sort_values(ascending=False)

        winner = ranking.index[0]

        second_votes = float(ranking.iloc[1]) if len(ranking) > 1 else 0.0

        smd_counts[winner] += 1

        for block in block_cols:

            votes = float(row[block])

            if block == winner:

                compensation[block] += max(0.0, votes - second_votes - 1.0)

            else:

                compensation[block] += votes



    adjusted_votes = {block: national_list_votes.get(block, 0.0) + compensation.get(block, 0.0) for block in block_cols}

    list_counts = allocate_dhondt(adjusted_votes, list_seats, threshold=0.05)

    total_counts = {block: smd_counts.get(block, 0) + list_counts.get(block, 0) for block in block_cols}

    return {"smd": smd_counts, "list": list_counts, "total": total_counts, "adjusted_votes": adjusted_votes}



print("Poll helper functions defined.")

Poll helper functions defined.


Interpretation. Lower coverage means the variable is harder to use safely without imputation.


## LOEO Validation Loop

For each of the four historical elections we hold it out, do all model selection on the remaining three elections (inner LOEO), and then predict the held-out election.

In [10]:
# this block evaluates how the model performed on past elections.
fold_results = []
selection_failure_rows = []

poll_ridge_features = [f"alr_{block}" for block in nonbase_blocks] + [
    "days_to_election",
    "undecided",
    "is_biztos",
]
poll_ridge_cat_specs = [("pollster", 5), ("mode", 5), ("election_type", 1)]
poll_ridge_targets = [f"actual_alr_{block}" for block in nonbase_blocks]

for hold_year, hold_type in LOEO_ELECTIONS:
    hold_election = f"{hold_year}_{hold_type}"
    train_elections = [(y, t) for y, t in LOEO_ELECTIONS if not (y == hold_year and t == hold_type)]
    train_election_names = [f"{y}_{t}" for y, t in train_elections]
    fold_warnings = []
    print(f"\n{'='*60}")
    print(f"LOEO fold: hold out {hold_election}")
    print(f"  training elections: {train_election_names}")

    # ── Poll data for this fold ────────────────────────────────────────
    fold_poll_all = poll_df[poll_df["election"].isin(train_election_names + [hold_election])].copy()
    fold_poll_train = fold_poll_all[fold_poll_all["election"].isin(train_election_names)].copy()
    fold_poll_test = fold_poll_all[fold_poll_all["election"] == hold_election].copy()

    # ── Inner LOEO: poll ridge alpha selection ─────────────────────────
    # For each alpha: for each of the 3 training elections as inner holdout,
    # fit ridge on remaining 2, predict inner holdout, compute MAE.
    # Select alpha with lowest mean MAE.
    fixed_poll_ridge_alpha = FIXED_POLL_RIDGE_ALPHA
    X_poll_full, _, _ = build_design(fold_poll_train, fold_poll_train, poll_ridge_features, poll_ridge_cat_specs)
    w_poll_full = fold_poll_train["sample_size"].fillna(1.0).to_numpy(dtype="float64")
    fixed_poll_ridge_alpha_cond = build_condition_numbers(X_poll_full.to_numpy(dtype="float64"), w_poll_full, [fixed_poll_ridge_alpha])[0]
    print(f"  poll ridge alpha: {fixed_poll_ridge_alpha} (cond: {fixed_poll_ridge_alpha_cond:.2e})")

    # ── Inner LOEO: poll model selection ──────────────────────────────
    # Candidates: simple_weighted_average, simple_average_plus_bias_floor, ridge_poll_model_05
    poll_model_inner_rows = []

    for inner_year, inner_type in train_elections:
        inner_election = f"{inner_year}_{inner_type}"
        inner_train = fold_poll_train[fold_poll_train["election"] != inner_election].copy()
        inner_test = fold_poll_train[fold_poll_train["election"] == inner_election].copy()
        if inner_train.empty or inner_test.empty:
            continue

        # simple_weighted_average
        simple_summary = summarize_simple_polls(inner_test, inner_election, simple_poll_window_days, actual_national_wide)
        if not simple_summary.empty:
            simple_pred = {
                block: float(simple_summary.loc[simple_summary["block"] == block, "mean"].iloc[0])
                if (simple_summary["block"] == block).any() else 0.25
                for block in block_cols
            }
            simple_row = summarize_poll_candidate(inner_election, simple_pred, actual_national_wide)
            poll_model_inner_rows.append({
                "model": "simple_weighted_average",
                "inner_holdout": inner_election,
                "mae": simple_row["mae"],
                "rmse": simple_row["rmse"],
            })
            poll_model_inner_rows.append({
                "model": "simple_average_plus_bias_floor",
                "inner_holdout": inner_election,
                "mae": simple_row["mae"],  # same point forecast, only uncertainty differs
                "rmse": simple_row["rmse"],
            })

        # ridge_poll_model_05
        try:
            ridge_bundle = fit_ridge_bundle(
                inner_train,
                poll_ridge_targets,
                poll_ridge_features,
                poll_ridge_cat_specs,
                "sample_size",
                alpha=fixed_poll_ridge_alpha,
            )
            pred_alr = predict_ridge_bundle(ridge_bundle, inner_test)
            pred_comp = inverse_alr(pred_alr, block_cols)
            pred_comp.columns = [f"ridge_pred_{block}" for block in block_cols]
            tmp = pd.concat([inner_test.reset_index(drop=True), pred_comp.reset_index(drop=True)], axis=1)
            tmp["agg_weight"] = tmp["sample_size"] * np.exp(-tmp["days_to_election"] / 60.0)
            ridge_pred_share = aggregate_poll_rows_to_share(
                tmp,
                {block: f"ridge_pred_{block}" for block in block_cols},
                "agg_weight",
            )
            ridge_row = summarize_poll_candidate(inner_election, ridge_pred_share, actual_national_wide)
            poll_model_inner_rows.append({
                "model": "ridge_poll_model_05",
                "inner_holdout": inner_election,
                "mae": ridge_row["mae"],
                "rmse": ridge_row["rmse"],
            })
        except Exception as e:
            error_text = f"{type(e).__name__}: {e}"
            poll_model_inner_rows.append({
                "model": "ridge_poll_model_05",
                "inner_holdout": inner_election,
                "mae": np.inf,
                "rmse": np.inf,
            })
            selection_failure_rows.append({
                "stage": "poll_model_inner",
                "hold_election": hold_election,
                "candidate": "ridge_poll_model_05",
                "inner_holdout": inner_election,
                "error": error_text,
            })

        # mixed_house_effect_only
        try:
            he_inner = fit_house_effects(
                inner_train, nonbase_blocks,
                house_effects_numeric_features, house_effects_cat_specs, "sample_size"
            )
            inner_test_corr = apply_house_effects(inner_test, he_inner, nonbase_blocks)
            inner_test_corr["agg_weight"] = inner_test_corr["sample_size"] * np.exp(-inner_test_corr["days_to_election"] / 60.0)
            house_only_pred_alr = {
                block: safe_weighted_average(inner_test_corr[f"alr_corr_{block}"], inner_test_corr["agg_weight"])
                for block in nonbase_blocks
            }
            house_only_comp = inverse_alr(
                np.array([[house_only_pred_alr[block] for block in nonbase_blocks]], dtype="float64"), block_cols
            ).iloc[0].to_dict()
            house_only_row = summarize_poll_candidate(inner_election, house_only_comp, actual_national_wide)
            poll_model_inner_rows.append({
                "model": "mixed_house_effect_only",
                "inner_holdout": inner_election,
                "mae": house_only_row["mae"],
                "rmse": house_only_row["rmse"],
            })
        except Exception as e:
            error_text = f"{type(e).__name__}: {e}"
            poll_model_inner_rows.append({
                "model": "mixed_house_effect_only",
                "inner_holdout": inner_election,
                "mae": np.inf,
                "rmse": np.inf,
            })
            selection_failure_rows.append({
                "stage": "poll_model_inner",
                "hold_election": hold_election,
                "candidate": "mixed_house_effect_only",
                "inner_holdout": inner_election,
                "error": error_text,
            })

        # house_effects_rw
        try:
            he_inner = fit_house_effects(
                inner_train, nonbase_blocks,
                house_effects_numeric_features, house_effects_cat_specs, "sample_size"
            )
            inner_test_corr = apply_house_effects(inner_test, he_inner, nonbase_blocks)
            rw_alr = run_random_walk_filter(
                inner_test_corr, nonbase_blocks, "sample_size",
                "days_to_election", rw_penalty_const
            )
            if any(np.isnan(v) for v in rw_alr.values()):
                raise ValueError("NaN in RW output")
            rw_comp = inverse_alr(
                np.array([[rw_alr.get(b, 0.0) for b in nonbase_blocks]]), block_cols
            ).iloc[0].to_dict()
            rw_row = summarize_poll_candidate(inner_election, rw_comp, actual_national_wide)
            poll_model_inner_rows.append({
                "model": "house_effects_rw",
                "inner_holdout": inner_election,
                "mae": rw_row["mae"],
                "rmse": rw_row["rmse"],
            })
        except Exception as e:
            error_text = f"{type(e).__name__}: {e}"
            poll_model_inner_rows.append({
                "model": "house_effects_rw",
                "inner_holdout": inner_election,
                "mae": np.inf,
                "rmse": np.inf,
            })
            selection_failure_rows.append({
                "stage": "poll_model_inner",
                "hold_election": hold_election,
                "candidate": "house_effects_rw",
                "inner_holdout": inner_election,
                "error": error_text,
            })

    # Aggregate inner LOEO results and select poll model
    if poll_model_inner_rows:
        poll_model_inner_df = pd.DataFrame(poll_model_inner_rows)
        poll_model_selection_df = (
            poll_model_inner_df
            .groupby("model", as_index=False)
            .agg(mean_mae=("mae", "mean"), mean_rmse=("rmse", "mean"))
        )
        selected_poll_model = choose_strict_poll_model(
            poll_model_selection_df,
            mae_tol=mae_tol,
        )
    else:
        selected_poll_model = "simple_weighted_average"
    print(f"  selected poll model: {selected_poll_model}")

    # ── Fit final poll model on all 3 training elections ──────────────
    # Then predict the held-out election
    if selected_poll_model == "ridge_poll_model_05":
        try:
            final_poll_bundle = fit_ridge_bundle(
                fold_poll_train,
                poll_ridge_targets,
                poll_ridge_features,
                poll_ridge_cat_specs,
                "sample_size",
                alpha=fixed_poll_ridge_alpha,
            )
            pred_alr_test = predict_ridge_bundle(final_poll_bundle, fold_poll_test)
            pred_comp_test = inverse_alr(pred_alr_test, block_cols)
            pred_comp_test.columns = [f"ridge_pred_{block}" for block in block_cols]
            fold_poll_test_tmp = pd.concat([fold_poll_test.reset_index(drop=True), pred_comp_test.reset_index(drop=True)], axis=1)
            fold_poll_test_tmp["agg_weight"] = fold_poll_test_tmp["sample_size"] * np.exp(-fold_poll_test_tmp["days_to_election"] / 60.0)
            fold_poll_forecast = aggregate_poll_rows_to_share(
                fold_poll_test_tmp,
                {block: f"ridge_pred_{block}" for block in block_cols},
                "agg_weight",
            )
        except Exception as e:
            fold_warnings.append(f"ridge poll model failed: {e}, falling back to simple")
            selected_poll_model = "simple_weighted_average"
            simple_sub = fold_poll_test[fold_poll_test["days_to_election"] <= simple_poll_window_days].copy()
            comp = inverse_alr(simple_sub[[f"alr_{block}" for block in nonbase_blocks]].to_numpy(dtype="float64"), block_cols)
            fold_poll_forecast = {
                block: safe_weighted_average(comp[block].to_numpy(), simple_sub["sample_size"].to_numpy())
                for block in block_cols
            }
            fold_poll_forecast = close_comp(pd.DataFrame([fold_poll_forecast]), block_cols, eps=composition_floor).iloc[0].to_dict()
    elif selected_poll_model == "mixed_house_effect_only":
        try:
            he_final = fit_house_effects(
                fold_poll_train, nonbase_blocks,
                house_effects_numeric_features, house_effects_cat_specs, "sample_size"
            )
            test_corr = apply_house_effects(fold_poll_test, he_final, nonbase_blocks)
            test_corr["agg_weight"] = test_corr["sample_size"] * np.exp(-test_corr["days_to_election"] / 60.0)
            house_only_pred_alr = {
                block: safe_weighted_average(test_corr[f"alr_corr_{block}"], test_corr["agg_weight"])
                for block in nonbase_blocks
            }
            fold_poll_forecast = inverse_alr(
                np.array([[house_only_pred_alr[block] for block in nonbase_blocks]], dtype="float64"), block_cols
            ).iloc[0].to_dict()
            fold_poll_forecast = close_comp(
                pd.DataFrame([fold_poll_forecast]), block_cols, eps=composition_floor
            ).iloc[0].to_dict()
        except Exception as e:
            fold_warnings.append(f"mixed_house_effect_only failed: {e}, falling back to simple")
            selected_poll_model = "simple_weighted_average"
            simple_sub = fold_poll_test[fold_poll_test["days_to_election"] <= simple_poll_window_days].copy()
            if simple_sub.empty:
                simple_sub = fold_poll_test.copy()
            comp = inverse_alr(simple_sub[[f"alr_{block}" for block in nonbase_blocks]].to_numpy(dtype="float64"), block_cols)
            fold_poll_forecast = {
                block: safe_weighted_average(comp[block].to_numpy(), simple_sub["sample_size"].to_numpy())
                for block in block_cols
            }
            fold_poll_forecast = close_comp(pd.DataFrame([fold_poll_forecast]), block_cols, eps=composition_floor).iloc[0].to_dict()
    elif selected_poll_model == "house_effects_rw":
        try:
            he_final = fit_house_effects(
                fold_poll_train, nonbase_blocks,
                house_effects_numeric_features, house_effects_cat_specs, "sample_size"
            )
            test_corr = apply_house_effects(fold_poll_test, he_final, nonbase_blocks)
            rw_alr = run_random_walk_filter(
                test_corr, nonbase_blocks, "sample_size",
                "days_to_election", rw_penalty_const
            )
            if any(np.isnan(v) for v in rw_alr.values()):
                raise ValueError("NaN in RW output")
            fold_poll_forecast = inverse_alr(
                np.array([[rw_alr.get(b, 0.0) for b in nonbase_blocks]]), block_cols
            ).iloc[0].to_dict()
            fold_poll_forecast = close_comp(
                pd.DataFrame([fold_poll_forecast]), block_cols, eps=composition_floor
            ).iloc[0].to_dict()
        except Exception as e:
            fold_warnings.append(f"house_effects_rw failed: {e}, falling back to simple")
            selected_poll_model = "simple_weighted_average"
            simple_sub = fold_poll_test[fold_poll_test["days_to_election"] <= simple_poll_window_days].copy()
            if simple_sub.empty:
                simple_sub = fold_poll_test.copy()
            comp = inverse_alr(simple_sub[[f"alr_{block}" for block in nonbase_blocks]].to_numpy(dtype="float64"), block_cols)
            fold_poll_forecast = {
                block: safe_weighted_average(comp[block].to_numpy(), simple_sub["sample_size"].to_numpy())
                for block in block_cols
            }
            fold_poll_forecast = close_comp(pd.DataFrame([fold_poll_forecast]), block_cols, eps=composition_floor).iloc[0].to_dict()
    else:
        # simple_weighted_average or simple_average_plus_bias_floor
        simple_sub = fold_poll_test[fold_poll_test["days_to_election"] <= simple_poll_window_days].copy()
        if simple_sub.empty:
            simple_sub = fold_poll_test.copy()
        comp = inverse_alr(simple_sub[[f"alr_{block}" for block in nonbase_blocks]].to_numpy(dtype="float64"), block_cols)
        fold_poll_forecast = {
            block: safe_weighted_average(comp[block].to_numpy(), simple_sub["sample_size"].to_numpy())
            for block in block_cols
        }
        fold_poll_forecast = close_comp(pd.DataFrame([fold_poll_forecast]), block_cols, eps=composition_floor).iloc[0].to_dict()

    # ── Prior layer: feature set selection (inner LOEO) ───────────────
    fold_prior_base = prior_base[prior_base["election"].isin(train_election_names)].copy()

    # Inner LOEO over 3 training elections for each model candidate
    prior_structure_inner_rows = []
    for model_name, model_features in preliminary_fundamentals_models.items():
        valid_features = [f for f in model_features if f in fold_prior_base.columns]
        for inner_year, inner_type in train_elections:
            inner_election = f"{inner_year}_{inner_type}"
            inner_train = fold_prior_base[~((fold_prior_base["year"] == inner_year) & (fold_prior_base["election_type"] == inner_type))].copy()
            inner_test = fold_prior_base[(fold_prior_base["year"] == inner_year) & (fold_prior_base["election_type"] == inner_type)].copy()
            if inner_train.empty or inner_test.empty or not valid_features:
                continue
            try:
                bundle = fit_ridge_bundle(
                    inner_train,
                    [f"target_alr_{block}" for block in nonbase_blocks],
                    valid_features,
                    prior_cat_specs,
                    "fit_weight",
                    alpha=FIXED_PRIOR_ALPHA,
                )
                pred_alr = predict_ridge_bundle(bundle, inner_test)
                pred_comp = inverse_alr(pred_alr, block_cols)
                weights = inner_test["fit_weight"].to_numpy(dtype="float64")
                actual_row = actual_national_wide[actual_national_wide["election"] == inner_election]
                pred_nat = {block: safe_weighted_average(pred_comp[block], weights) for block in block_cols}
                nat_mae = float(np.mean([abs(pred_nat[b] - float(actual_row.iloc[0][b])) for b in block_cols])) if not actual_row.empty else np.inf
                prior_structure_inner_rows.append({
                    "scenario": model_name,
                    "inner_holdout": inner_election,
                    "national_mae": nat_mae,
                })
            except Exception as e:
                error_text = f"{type(e).__name__}: {e}"
                prior_structure_inner_rows.append({
                    "scenario": model_name,
                    "inner_holdout": inner_election,
                    "national_mae": np.inf,
                })
                selection_failure_rows.append({
                    "stage": "prior_structure_inner",
                    "hold_election": hold_election,
                    "candidate": model_name,
                    "inner_holdout": inner_election,
                    "error": error_text,
                })

    if prior_structure_inner_rows:
        prior_struct_df = pd.DataFrame(prior_structure_inner_rows).groupby("scenario", as_index=False).agg(mean_national_mae=("national_mae", "mean"))
        best_prior_structure = str(prior_struct_df.sort_values("mean_national_mae").iloc[0]["scenario"])
    else:
        best_prior_structure = list(preliminary_fundamentals_models.keys())[0]
    best_prior_structure_features = [f for f in preliminary_fundamentals_models.get(best_prior_structure, []) if f in fold_prior_base.columns]
    print(f"  selected prior structure: {best_prior_structure}")

    # Always add share_lag_features + selected_recent_media_features on top
    prior_features_full = list(dict.fromkeys(best_prior_structure_features + share_lag_features + selected_recent_media_features))
    prior_features_screened, _, _ = screen_sparse_features(
        fold_prior_base,
        prior_features_full,
        min_nonmissing_share=full_history_min_nonmissing_share,
        grouped_prefixes=[("has_lag_nat", "lag_nat_"), ("has_lag_ep", "lag_ep_")],
    )
    if not prior_features_screened:
        prior_features_screened = best_prior_structure_features.copy()

    # Fixed alpha (hardcoded based on MAE curve analysis)
    fixed_prior_alpha = FIXED_PRIOR_ALPHA
    X_prior_full, _, _ = build_design(fold_prior_base, fold_prior_base, prior_features_screened, prior_cat_specs)
    w_prior_full = fold_prior_base["fit_weight"].fillna(1.0).to_numpy(dtype="float64")
    fixed_prior_alpha_cond = build_condition_numbers(X_prior_full.to_numpy(dtype="float64"), w_prior_full, [fixed_prior_alpha])[0]
    print(f"  prior alpha: {fixed_prior_alpha} (cond: {fixed_prior_alpha_cond:.2e})")

    # ── Fit final prior model on all 3 training elections ─────────────
    # Predict the held-out election districts
    fold_test_districts = prior_base[(prior_base["year"] == hold_year) & (prior_base["election_type"] == hold_type)].copy()
    # Use only pre-election registered voters as weights for the held-out fold.
    # valid_list_total is post-election data; electorate_list is available before the election.
    fold_test_districts["fit_weight"] = fold_test_districts["electorate_list"].fillna(1.0)
    if fold_test_districts.empty:
        fold_warnings.append(f"No district data for held-out election {hold_election}")
        fold_results.append({
            "hold_election": hold_election,
            "hold_year": hold_year,
            "hold_type": hold_type,
            "national_mae": np.nan,
            "district_mae": np.nan,
            "selected_poll_model": selected_poll_model,
            "fixed_poll_ridge_alpha": fixed_poll_ridge_alpha,
            "selected_prior_structure": best_prior_structure,
            "fixed_prior_alpha": fixed_prior_alpha,
            "fixed_lean_alpha": np.nan,
            "rule_based_correction_model": "none",
            "warnings": "; ".join(fold_warnings),
        })
        continue

    weights_test = fold_test_districts["fit_weight"].to_numpy(dtype="float64")

    pred_comp_districts = None

    try:
        final_prior_bundle = fit_ridge_bundle(
            fold_prior_base,
            [f"target_alr_{block}" for block in nonbase_blocks],
            prior_features_screened,
            prior_cat_specs,
            "fit_weight",
            alpha=fixed_prior_alpha,
        )
        pred_alr_districts = predict_ridge_bundle(final_prior_bundle, fold_test_districts)
        pred_comp_districts = inverse_alr(pred_alr_districts, block_cols)
        for block in block_cols:
            fold_test_districts[f"pred_{block}"] = pred_comp_districts[block].to_numpy()
        prior_nat = {block: safe_weighted_average(fold_test_districts[f"pred_{block}"], weights_test) for block in block_cols}
        prior_forecast_alr = alr(pd.DataFrame([prior_nat]), block_cols).iloc[0].to_numpy(dtype="float64")
    except Exception as e:
        fold_warnings.append(f"prior model failed: {e}")
        prior_nat = {block: 0.25 for block in block_cols}
        prior_forecast_alr = alr(pd.DataFrame([prior_nat]), block_cols).iloc[0].to_numpy(dtype="float64")

    prior_mixed_pred_comp = None

    try:
        prior_mixed_pred_parts = []

        for block in nonbase_blocks:

            prior_mixed_model = fit_random_intercept(
                fold_prior_base,
                f"target_alr_{block}",
                prior_features_screened,
                prior_cat_specs,
                "county",
            )

            prior_mixed_pred_parts.append(predict_random_intercept(prior_mixed_model, fold_test_districts))

        prior_mixed_pred_comp = inverse_alr(np.column_stack(prior_mixed_pred_parts), block_cols)

    except Exception as e:

        fold_warnings.append(f"prior mixed comparison failed: {e}")

    # ── Lean layer: alpha selection (inner LOEO) ───────────────────────
    fold_lean_train_hist = lean_hist[lean_hist["election"].isin(train_election_names)].copy()
    fold_lean_train = add_lean_lags(fold_lean_train_hist)

    # Build lean feature set
    # Base district lean is structural only. Lagged lean belongs to the recent correction layer.
    lean_features_raw = screened_full_history_structure.copy()
    lean_features_screened, _, _ = screen_sparse_features(
        fold_lean_train,
        lean_features_raw,
        min_nonmissing_share=full_history_min_nonmissing_share,
    )
    if not lean_features_screened:
        lean_features_screened = screened_full_history_structure.copy()

    fixed_lean_alpha = FIXED_LEAN_ALPHA
    X_lean_full, _, _ = build_design(fold_lean_train, fold_lean_train, lean_features_screened, prior_cat_specs)
    w_lean_full = fold_lean_train["valid_list_total"].fillna(1.0).to_numpy(dtype="float64")
    fixed_lean_alpha_cond = build_condition_numbers(X_lean_full.to_numpy(dtype="float64"), w_lean_full, [fixed_lean_alpha])[0]
    print(f"  lean alpha: {fixed_lean_alpha} (cond: {fixed_lean_alpha_cond:.2e})")

    # ── Fit final lean model on 3 training elections ───────────────────
    fold_lean_test_hist = lean_hist[(lean_hist["year"] == hold_year) & (lean_hist["election_type"] == hold_type)].copy()
    fold_lean_test = add_lean_lags(fold_lean_test_hist)

    try:
        final_lean_bundle = fit_ridge_bundle(
            fold_lean_train,
            [f"lean_alr_{block}" for block in nonbase_blocks],
            lean_features_screened,
            prior_cat_specs,
            "valid_list_total",
            alpha=fixed_lean_alpha,
        )
        pred_lean_test = predict_ridge_bundle(final_lean_bundle, fold_lean_test)
        for idx, block in enumerate(nonbase_blocks):
            fold_lean_test[f"pred_lean_alr_{block}"] = pred_lean_test[:, idx]
    except Exception as e:
        fold_warnings.append(f"lean model failed: {e}")
        for block in nonbase_blocks:
            fold_lean_test[f"pred_lean_alr_{block}"] = 0.0

    lean_ridge_share_base = inverse_alr(
        np.column_stack([
            fold_lean_test[f"nat_alr_{block}"].to_numpy(dtype="float64") + fold_lean_test[f"pred_lean_alr_{block}"].to_numpy(dtype="float64")
            for block in nonbase_blocks
        ]),
        block_cols,
    )

    lean_mixed_share_base = None

    try:
        lean_mixed_pred_parts = []

        for block in nonbase_blocks:

            lean_mixed_model = fit_random_intercept(
                fold_lean_train,
                f"lean_alr_{block}",
                lean_features_screened,
                prior_cat_specs,
                "county",
            )

            lean_mixed_pred_parts.append(predict_random_intercept(lean_mixed_model, fold_lean_test))

        lean_mixed_share_base = inverse_alr(
            np.column_stack([
                fold_lean_test[f"nat_alr_{block}"].to_numpy(dtype="float64") + lean_mixed_pred_parts[idx]
                for idx, block in enumerate(nonbase_blocks)
            ]),
            block_cols,
        )

    except Exception as e:

        fold_warnings.append(f"lean mixed comparison failed: {e}")

    # ── Recent correction: feature screening and model assignment ────────
    # Only 2022 and 2024 have the needed data for the correction layer.
    # The correction model is hardcoded to lag_plus_media based on theory;
    # data-driven selection is not used because with at most 1 training
    # election per inner fold the result would be too noisy to be reliable.
    recent_correction_years = [y for y in [2022, 2024] if y != hold_year]
    if len(recent_correction_years) < 2:
        fold_warnings.append(f"Only {len(recent_correction_years)} recent correction year(s) available (held out {hold_year})")

    recent_correction_elections_fold = [f"{y}_{'nat' if y == 2022 else 'ep'}" for y in recent_correction_years]
    fold_recent_train = fold_lean_train[fold_lean_train["year"].isin(recent_correction_years)].copy()
    fold_recent_train["election"] = fold_recent_train["year"].astype(str) + "_" + fold_recent_train["election_type"]

    # Correction model: hardcoded lag_plus_media based on theory and literature.
    # Persistent spatial voting patterns (lag lean) and documented Hungarian media bias
    # are well-established; with at most 1 training election per inner fold, data-driven
    # selection would be too noisy to be meaningful.
    # For 2018/2019 test folds, lag features may be NaN (2014 data unavailable);
    # those NaN columns are dropped so the correction falls back to media-only.
    rule_based_correction_model = "none"
    selected_correction_features = []

    if not fold_recent_train.empty and recent_correction_years:
        # Screen lag and media features on the training data (2022/2024 rows)
        lag_correction_features = (
            [f"lag_nat_lean_{block}" for block in nonbase_blocks]
            + [f"lag_ep_lean_{block}" for block in nonbase_blocks]
        )
        lag_correction_features_screened, _, _ = screen_sparse_features(
            fold_recent_train,
            lag_correction_features,
            min_nonmissing_share=full_history_min_nonmissing_share,
        )
        media_correction_features_screened, _, _ = screen_sparse_features(
            fold_recent_train,
            [f for f in selected_recent_media_features if f in fold_recent_train.columns],
            min_nonmissing_share=full_history_min_nonmissing_share,
        ) if selected_recent_media_features else ([], [], pd.DataFrame())

        # Also screen lag features on the test data: 2018/2019 folds may lack 2014 lags
        if lag_correction_features_screened and not fold_lean_test.empty:
            lag_test_ok, _, _ = screen_sparse_features(
                fold_lean_test,
                lag_correction_features_screened,
                min_nonmissing_share=full_history_min_nonmissing_share,
            )
        else:
            lag_test_ok = lag_correction_features_screened

        # Build feature set: lag_plus_media where both available in test; else media_only
        if lag_test_ok and media_correction_features_screened:
            rule_based_correction_model = "lag_plus_media"
            selected_correction_features = list(
                dict.fromkeys(lag_test_ok + media_correction_features_screened)
            )
        elif lag_test_ok:
            rule_based_correction_model = "lag_only"
            selected_correction_features = lag_test_ok
        elif media_correction_features_screened:
            rule_based_correction_model = "media_only"
            selected_correction_features = media_correction_features_screened

    print(f"  correction model by rule: {rule_based_correction_model} (hardcoded by theory)")

    recent_correction_applied = False
    if selected_correction_features and not fold_recent_train.empty:
        try:
            base_pred_recent = predict_ridge_bundle(final_lean_bundle, fold_recent_train)
            for idx, block in enumerate(nonbase_blocks):
                fold_recent_train[f"recent_correction_target_{block}"] = (
                    fold_recent_train[f"lean_alr_{block}"].to_numpy(dtype="float64") - base_pred_recent[:, idx]
                )

            recent_correction_bundle = fit_ridge_bundle(
                fold_recent_train,
                [f"recent_correction_target_{block}" for block in nonbase_blocks],
                selected_correction_features,
                recent_correction_cat_specs,
                "valid_list_total",
                alpha=recent_correction_alpha,
            )
            correction_pred_test = predict_ridge_bundle(recent_correction_bundle, fold_lean_test)
            for idx, block in enumerate(nonbase_blocks):
                fold_lean_test[f"pred_lean_alr_{block}"] = (
                    fold_lean_test[f"pred_lean_alr_{block}"].to_numpy(dtype="float64")
                    + 1.0 * correction_pred_test[:, idx]
                )
            recent_correction_applied = True
        except Exception as e:
            fold_warnings.append(f"Recent correction failed: {e}")

    # ── National combination: FIXED_POLL_WEIGHT = 0.70 ────────────────
    poll_share = close_comp(pd.DataFrame([fold_poll_forecast]), block_cols, eps=composition_floor).iloc[0].to_dict()
    prior_share = close_comp(pd.DataFrame([prior_nat]), block_cols, eps=composition_floor).iloc[0].to_dict()
    combined_share = close_comp(
        pd.DataFrame([{
            block: FIXED_POLL_WEIGHT * float(poll_share[block]) + (1.0 - FIXED_POLL_WEIGHT) * float(prior_share[block])
            for block in block_cols
        }]),
        block_cols,
        eps=composition_floor,
    ).iloc[0].to_dict()
    combined_alr = alr(pd.DataFrame([combined_share]), block_cols).iloc[0].to_numpy(dtype="float64")

    # ── Align district leans to national combined ──────────────────────
    predicted_lean_matrix = fold_lean_test[[f"pred_lean_alr_{block}" for block in nonbase_blocks]].to_numpy(dtype="float64")
    list_forecast_weights = fold_lean_test["electorate_list"].fillna(1.0).to_numpy(dtype="float64")

    district_alr_aligned = align_local_alr_to_national(
        predicted_lean_matrix,
        combined_alr,
        list_forecast_weights,
        block_cols,
    )
    district_comp_aligned = inverse_alr(district_alr_aligned, block_cols)
    for block in block_cols:
        fold_lean_test[f"combined_{block}"] = district_comp_aligned[block].to_numpy()

    # ── Compute errors ─────────────────────────────────────────────────
    # National MAE: combined forecast vs actual national shares for held-out election
    actual_hold_row = actual_national_wide[actual_national_wide["election"] == hold_election]
    if not actual_hold_row.empty:
        actual_hold_share = {block: float(actual_hold_row.iloc[0][block]) for block in block_cols}
        national_mae = float(np.mean([abs(combined_share[b] - actual_hold_share[b]) for b in block_cols]))
        national_mae_gov_opp_opprad = float(np.mean([abs(combined_share[b] - actual_hold_share[b]) for b in nonbase_blocks]))
    else:
        actual_hold_share = {block: np.nan for block in block_cols}
        national_mae = np.nan
        national_mae_gov_opp_opprad = np.nan

    # District MAE: predicted district shares vs actual district list_shares
    # Computed across gov, opp, opp_radical only.
    # fold_lean_test inherits list_share_* from lean_hist, so select only the
    # oevk_id and combined_* columns before merging to avoid _x/_y name conflicts.
    hold_result_wide = result_wide[(result_wide["year"] == hold_year) & (result_wide["election_type"] == hold_type)].copy()
    district_mae_list = []
    actual_cols_exist = all(f"list_share_{block}" in hold_result_wide.columns for block in nonbase_blocks)
    if not hold_result_wide.empty and not fold_lean_test.empty and actual_cols_exist:
        merged_district = (
            fold_lean_test[["oevk_id"] + [f"combined_{block}" for block in nonbase_blocks]]
            .merge(
                hold_result_wide[["oevk_id"] + [f"list_share_{block}" for block in nonbase_blocks]],
                on="oevk_id",
                how="inner",
            )
        )
        if not merged_district.empty:
            for block in nonbase_blocks:
                errs = np.abs(
                    merged_district[f"combined_{block}"].to_numpy(dtype="float64")
                    - merged_district[f"list_share_{block}"].to_numpy(dtype="float64")
                )
                district_mae_list.append(float(errs.mean()))
    district_mae = float(np.mean(district_mae_list)) if district_mae_list else np.nan

    prior_ridge_national_mae_all_blocks = np.nan
    prior_ridge_district_mae_all_blocks = np.nan
    prior_mixed_national_mae_all_blocks = np.nan
    prior_mixed_district_mae_all_blocks = np.nan
    lean_ridge_district_mae_all_blocks = np.nan
    lean_ridge_winner_hit_rate = np.nan
    lean_mixed_district_mae_all_blocks = np.nan
    lean_mixed_winner_hit_rate = np.nan
    seat_total_abs_error = np.nan
    pred_seats = {block: np.nan for block in block_cols}
    target_seats = {block: np.nan for block in block_cols}

    all_block_actual_cols_exist = all(f"list_share_{block}" in hold_result_wide.columns for block in block_cols)

    if pred_comp_districts is not None:

        prior_ridge_nat = {
            block: safe_weighted_average(fold_test_districts[f"pred_{block}"], weights_test)
            for block in block_cols
        }

        if not actual_hold_row.empty:

            prior_ridge_national_mae_all_blocks = float(np.mean([abs(prior_ridge_nat[block] - actual_hold_share[block]) for block in block_cols]))

        if not hold_result_wide.empty and all_block_actual_cols_exist:

            merged_prior_ridge = (
                fold_test_districts[["oevk_id"] + [f"pred_{block}" for block in block_cols]]
                .merge(
                    hold_result_wide[["oevk_id"] + [f"list_share_{block}" for block in block_cols]],
                    on="oevk_id",
                    how="inner",
                )
            )

            if not merged_prior_ridge.empty:

                prior_ridge_district_mae_all_blocks = float(np.mean([
                    np.abs(merged_prior_ridge[f"pred_{block}"] - merged_prior_ridge[f"list_share_{block}"]).mean()
                    for block in block_cols
                ]))

    if prior_mixed_pred_comp is not None:

        prior_mixed_nat = {
            block: safe_weighted_average(prior_mixed_pred_comp[block], weights_test)
            for block in block_cols
        }

        if not actual_hold_row.empty:

            prior_mixed_national_mae_all_blocks = float(np.mean([abs(prior_mixed_nat[block] - actual_hold_share[block]) for block in block_cols]))

        if not hold_result_wide.empty and all_block_actual_cols_exist:

            prior_mixed_pred_df = fold_test_districts[["oevk_id"]].copy()

            for block in block_cols:

                prior_mixed_pred_df[f"pred_{block}"] = prior_mixed_pred_comp[block].to_numpy(dtype="float64")

            merged_prior_mixed = (
                prior_mixed_pred_df
                .merge(
                    hold_result_wide[["oevk_id"] + [f"list_share_{block}" for block in block_cols]],
                    on="oevk_id",
                    how="inner",
                )
            )

            if not merged_prior_mixed.empty:

                prior_mixed_district_mae_all_blocks = float(np.mean([
                    np.abs(merged_prior_mixed[f"pred_{block}"] - merged_prior_mixed[f"list_share_{block}"]).mean()
                    for block in block_cols
                ]))

    if not hold_result_wide.empty and all_block_actual_cols_exist:

        lean_actual_df = hold_result_wide[["oevk_id"] + [f"list_share_{block}" for block in block_cols]].copy()

        lean_ridge_pred_df = fold_lean_test[["oevk_id"]].copy()

        for block in block_cols:

            lean_ridge_pred_df[f"pred_{block}"] = lean_ridge_share_base[block].to_numpy(dtype="float64")

        merged_lean_ridge = lean_ridge_pred_df.merge(lean_actual_df, on="oevk_id", how="inner")

        if not merged_lean_ridge.empty:

            lean_ridge_district_mae_all_blocks = float(np.mean([
                np.abs(merged_lean_ridge[f"pred_{block}"] - merged_lean_ridge[f"list_share_{block}"]).mean()
                for block in block_cols
            ]))

            lean_ridge_winner_hit_rate = float((
                merged_lean_ridge[[f"pred_{block}" for block in block_cols]].idxmax(axis=1).str.replace("pred_", "", regex=False)
                == merged_lean_ridge[[f"list_share_{block}" for block in block_cols]].idxmax(axis=1).str.replace("list_share_", "", regex=False)
            ).mean())

        if lean_mixed_share_base is not None:

            lean_mixed_pred_df = fold_lean_test[["oevk_id"]].copy()

            for block in block_cols:

                lean_mixed_pred_df[f"pred_{block}"] = lean_mixed_share_base[block].to_numpy(dtype="float64")

            merged_lean_mixed = lean_mixed_pred_df.merge(lean_actual_df, on="oevk_id", how="inner")

            if not merged_lean_mixed.empty:

                lean_mixed_district_mae_all_blocks = float(np.mean([
                    np.abs(merged_lean_mixed[f"pred_{block}"] - merged_lean_mixed[f"list_share_{block}"]).mean()
                    for block in block_cols
                ]))

                lean_mixed_winner_hit_rate = float((
                    merged_lean_mixed[[f"pred_{block}" for block in block_cols]].idxmax(axis=1).str.replace("pred_", "", regex=False)
                    == merged_lean_mixed[[f"list_share_{block}" for block in block_cols]].idxmax(axis=1).str.replace("list_share_", "", regex=False)
                ).mean())

    if hold_type == "nat" and not fold_lean_test.empty:

        seat_scale = fold_lean_test["electorate_list"].fillna(1.0)

        district_list_votes_pred = fold_lean_test[[f"combined_{block}" for block in block_cols]].rename(columns={f"combined_{block}": block for block in block_cols}).multiply(seat_scale, axis=0)

        district_cand_votes_pred = district_list_votes_pred.copy()

        seat_point = legal_parliament_seats(district_list_votes_pred, district_cand_votes_pred, list_seats=93)

        actual_hold_nat = final_results_oevk_party[
            (final_results_oevk_party["year"] == hold_year)
            & (final_results_oevk_party["election_type"] == "nat")
        ].copy()

        target_list_votes = actual_hold_nat.groupby(["oevk_id", "party_block"], as_index=False)["list_votes"].sum().pivot(index="oevk_id", columns="party_block", values="list_votes").fillna(0.0)
        target_cand_votes = actual_hold_nat.groupby(["oevk_id", "party_block"], as_index=False)["cand_votes"].sum().pivot(index="oevk_id", columns="party_block", values="cand_votes").fillna(0.0)

        for block in block_cols:
            if block not in target_list_votes.columns:
                target_list_votes[block] = 0.0
                target_cand_votes[block] = 0.0

        target_list_votes = target_list_votes[block_cols]
        target_cand_votes = target_cand_votes[block_cols]
        target_point = legal_parliament_seats(target_list_votes, target_cand_votes, list_seats=93)

        pred_seats = {block: int(seat_point["total"].get(block, 0)) for block in block_cols}

        target_seats = {block: int(target_point["total"].get(block, 0)) for block in block_cols}

        seat_total_abs_error = float(sum(abs(pred_seats[block] - target_seats[block]) for block in block_cols))

    print(f"  national MAE (all blocks): {national_mae:.4f}")
    print(f"  national MAE (gov/opp/opp_rad): {national_mae_gov_opp_opprad:.4f}")
    print(f"  district MAE (gov/opp/opp_rad): {district_mae:.4f}")

    if np.isfinite(prior_ridge_national_mae_all_blocks) or np.isfinite(prior_mixed_national_mae_all_blocks):

        print(
            f"  prior national MAE ridge vs mixed: {prior_ridge_national_mae_all_blocks:.4f} vs {prior_mixed_national_mae_all_blocks:.4f}"
        )

    if np.isfinite(lean_ridge_district_mae_all_blocks) or np.isfinite(lean_mixed_district_mae_all_blocks):

        print(
            f"  lean district MAE ridge vs mixed: {lean_ridge_district_mae_all_blocks:.4f} vs {lean_mixed_district_mae_all_blocks:.4f}"
        )

    if np.isfinite(seat_total_abs_error):

        print(f"  seat total absolute error vs harmonized-map target: {seat_total_abs_error:.1f}")

    if fold_warnings:

        print(f"  warnings: {fold_warnings}")

    fold_results.append({
        "hold_election": hold_election,
        "hold_year": hold_year,
        "hold_type": hold_type,
        "national_mae_all_blocks": national_mae,
        "national_mae_gov_opp_opprad": national_mae_gov_opp_opprad,
        "district_mae_gov_opp_opprad": district_mae,
        "prior_ridge_national_mae_all_blocks": prior_ridge_national_mae_all_blocks,
        "prior_ridge_district_mae_all_blocks": prior_ridge_district_mae_all_blocks,
        "prior_mixed_national_mae_all_blocks": prior_mixed_national_mae_all_blocks,
        "prior_mixed_district_mae_all_blocks": prior_mixed_district_mae_all_blocks,
        "lean_ridge_district_mae_all_blocks": lean_ridge_district_mae_all_blocks,
        "lean_ridge_winner_hit_rate": lean_ridge_winner_hit_rate,
        "lean_mixed_district_mae_all_blocks": lean_mixed_district_mae_all_blocks,
        "lean_mixed_winner_hit_rate": lean_mixed_winner_hit_rate,
        "seat_total_abs_error": seat_total_abs_error,
        "selected_poll_model": selected_poll_model,
        "fixed_poll_ridge_alpha": fixed_poll_ridge_alpha,
        "selected_prior_structure": best_prior_structure,
        "fixed_prior_alpha": fixed_prior_alpha,
        "fixed_lean_alpha": fixed_lean_alpha,
        "poll_ridge_alpha_cond": fixed_poll_ridge_alpha_cond,
        "prior_alpha_cond": fixed_prior_alpha_cond,
        "lean_alpha_cond": fixed_lean_alpha_cond,
        "rule_based_correction_model": rule_based_correction_model,
        "recent_correction_applied": recent_correction_applied,
        "recent_correction_years_used": recent_correction_years,
        "poll_forecast_gov": fold_poll_forecast.get("gov", np.nan),
        "poll_forecast_opp": fold_poll_forecast.get("opp", np.nan),
        "prior_nat_gov": prior_nat.get("gov", np.nan),
        "prior_nat_opp": prior_nat.get("opp", np.nan),
        "combined_gov": combined_share.get("gov", np.nan),
        "combined_opp": combined_share.get("opp", np.nan),
        "actual_gov": actual_hold_share.get("gov", np.nan),
        "actual_opp": actual_hold_share.get("opp", np.nan),
        "pred_seats_gov": pred_seats.get("gov", np.nan),
        "pred_seats_opp": pred_seats.get("opp", np.nan),
        "pred_seats_opp_radical": pred_seats.get("opp_radical", np.nan),
        "pred_seats_other": pred_seats.get("other", np.nan),
        "target_seats_gov": target_seats.get("gov", np.nan),
        "target_seats_opp": target_seats.get("opp", np.nan),
        "target_seats_opp_radical": target_seats.get("opp_radical", np.nan),
        "target_seats_other": target_seats.get("other", np.nan),
        "warnings": "; ".join(fold_warnings) if fold_warnings else "",
    })


print("\nLOEO loop complete.")



LOEO fold: hold out 2018_nat
  training elections: ['2019_ep', '2022_nat', '2024_ep']


  poll ridge alpha: 20.0 (cond: 4.58e+03)


  selected poll model: simple_weighted_average
  selected prior structure: model_1_demography_only
  prior alpha: 100.0 (cond: 4.39e+05)
  lean alpha: 30.0 (cond: 2.48e+02)


  correction model by rule: media_only (hardcoded by theory)
  national MAE (all blocks): 0.0254
  national MAE (gov/opp/opp_rad): 0.0310
  district MAE (gov/opp/opp_rad): 0.0471
  prior national MAE ridge vs mixed: 0.0643 vs 0.0646
  lean district MAE ridge vs mixed: 0.0290 vs 0.0266
  seat total absolute error vs harmonized-map target: 20.0

LOEO fold: hold out 2019_ep
  training elections: ['2018_nat', '2022_nat', '2024_ep']
  poll ridge alpha: 20.0 (cond: 4.30e+04)


  selected poll model: simple_weighted_average
  selected prior structure: model_3_plus_modernization
  prior alpha: 100.0 (cond: 2.18e+02)
  lean alpha: 30.0 (cond: 2.41e+02)


  correction model by rule: lag_plus_media (hardcoded by theory)
  national MAE (all blocks): 0.0221
  national MAE (gov/opp/opp_rad): 0.0235
  district MAE (gov/opp/opp_rad): 0.0271
  prior national MAE ridge vs mixed: 0.0924 vs 0.0924
  lean district MAE ridge vs mixed: 0.0202 vs 0.0174

LOEO fold: hold out 2022_nat
  training elections: ['2018_nat', '2019_ep', '2024_ep']
  poll ridge alpha: 20.0 (cond: 3.01e+04)


  selected poll model: simple_weighted_average
  selected prior structure: model_2_plus_core_economy
  prior alpha: 100.0 (cond: 1.62e+02)


  lean alpha: 30.0 (cond: 2.37e+02)
  correction model by rule: lag_plus_media (hardcoded by theory)
  national MAE (all blocks): 0.0164
  national MAE (gov/opp/opp_rad): 0.0204
  district MAE (gov/opp/opp_rad): 0.0305
  prior national MAE ridge vs mixed: 0.0851 vs 0.0905
  lean district MAE ridge vs mixed: 0.0200 vs 0.0173
  seat total absolute error vs harmonized-map target: 16.0
  warnings: ['Only 1 recent correction year(s) available (held out 2022)']

LOEO fold: hold out 2024_ep
  training elections: ['2018_nat', '2019_ep', '2022_nat']
  poll ridge alpha: 20.0 (cond: 4.11e+04)


  selected poll model: simple_weighted_average
  selected prior structure: model_1_demography_only
  prior alpha: 100.0 (cond: 1.17e+02)
  lean alpha: 30.0 (cond: 2.46e+02)


  correction model by rule: lag_plus_media (hardcoded by theory)
  national MAE (all blocks): 0.0128
  national MAE (gov/opp/opp_rad): 0.0162
  district MAE (gov/opp/opp_rad): 0.0250
  prior national MAE ridge vs mixed: 0.0598 vs 0.0607
  lean district MAE ridge vs mixed: 0.0252 vs 0.0217
  warnings: ['Only 1 recent correction year(s) available (held out 2024)']

LOEO loop complete.


Interpretation. Lower error means the method matched past election results better.


## Curiosity Check: Recent Correction Variants

**Curiosity note:** This is only a side test for inspection. It does **not** change the LOEO
results above or any later notebook outputs. The block below reruns a small grouped-county
backtest on the recent-elections subset only (`2022_nat` and `2024_ep`) and compares four fixed
variants: no recent correction, lag only, media only, and lag + media.


In [11]:
# curiosity-only diagnostic: compare recent-correction variants without changing notebook state.
curiosity_global_guard_before = (
    len(fold_results),
    tuple(selected_recent_media_features),
    float(recent_correction_alpha),
)

curiosity_recent_years = [2022, 2024]
curiosity_recent_elections = ["2022_nat", "2024_ep"]
curiosity_lean_all = add_lean_lags(lean_hist[lean_hist["year"].isin([2018, 2019, 2022, 2024])].copy())
curiosity_recent_train = curiosity_lean_all[curiosity_lean_all["year"].isin(curiosity_recent_years)].copy()

curiosity_lean_features_raw = screened_full_history_structure.copy()
curiosity_lean_features, _, _ = screen_sparse_features(
    curiosity_lean_all,
    curiosity_lean_features_raw,
    min_nonmissing_share=full_history_min_nonmissing_share,
)
if not curiosity_lean_features:
    curiosity_lean_features = screened_full_history_structure.copy()

curiosity_lag_feature_candidates = (
    [f"lag_nat_lean_{block}" for block in nonbase_blocks]
    + [f"lag_ep_lean_{block}" for block in nonbase_blocks]
)
curiosity_lag_features, _, _ = screen_sparse_features(
    curiosity_recent_train,
    curiosity_lag_feature_candidates,
    min_nonmissing_share=full_history_min_nonmissing_share,
)
curiosity_media_feature_candidates = [
    feature for feature in selected_recent_media_features if feature in curiosity_recent_train.columns
]
curiosity_media_features, _, _ = screen_sparse_features(
    curiosity_recent_train,
    curiosity_media_feature_candidates,
    min_nonmissing_share=full_history_min_nonmissing_share,
) if curiosity_media_feature_candidates else ([], [], pd.DataFrame())

curiosity_variant_map = {
    "no_recent_correction": [],
    "lag_only": curiosity_lag_features,
    "media_only": curiosity_media_features,
    "lag_plus_media": list(dict.fromkeys(curiosity_lag_features + curiosity_media_features)),
}
curiosity_variant_rank = {
    "no_recent_correction": 0,
    "lag_only": 1,
    "media_only": 2,
    "lag_plus_media": 3,
}

curiosity_rows = []
for curiosity_fold_id, curiosity_holdout_groups in enumerate(grouped_balanced_folds(curiosity_recent_train, "county", n_folds=5), start=1):
    curiosity_holdout_mask = (
        curiosity_recent_train["county"]
        .astype("string")
        .fillna("Unknown")
        .astype(str)
        .isin(curiosity_holdout_groups)
    )
    curiosity_correction_train = curiosity_recent_train.loc[~curiosity_holdout_mask].copy()
    curiosity_correction_test = curiosity_recent_train.loc[curiosity_holdout_mask].copy()
    if curiosity_correction_train.empty or curiosity_correction_test.empty:
        continue

    curiosity_base_train = curiosity_lean_all[
        ~(
            curiosity_lean_all["year"].isin(curiosity_recent_years)
            & curiosity_lean_all["county"].astype("string").fillna("Unknown").astype(str).isin(curiosity_holdout_groups)
        )
    ].copy()
    if curiosity_base_train.empty:
        continue

    curiosity_base_bundle = fit_ridge_bundle(
        curiosity_base_train,
        [f"lean_alr_{block}" for block in nonbase_blocks],
        curiosity_lean_features,
        prior_cat_specs,
        "valid_list_total",
        alpha=FIXED_LEAN_ALPHA,
    )
    curiosity_base_pred_train = predict_ridge_bundle(curiosity_base_bundle, curiosity_correction_train)
    curiosity_base_pred_test = predict_ridge_bundle(curiosity_base_bundle, curiosity_correction_test)

    curiosity_targets_train = curiosity_correction_train.copy()
    for curiosity_idx, curiosity_block in enumerate(nonbase_blocks):
        curiosity_targets_train[f"recent_correction_target_{curiosity_block}"] = (
            curiosity_targets_train[f"lean_alr_{curiosity_block}"].to_numpy(dtype="float64")
            - curiosity_base_pred_train[:, curiosity_idx]
        )

    curiosity_actual_share = curiosity_correction_test[[f"list_share_{block}" for block in block_cols]].rename(
        columns={f"list_share_{block}": block for block in block_cols}
    ).reset_index(drop=True)
    curiosity_election_labels = curiosity_correction_test["year"].astype(str) + "_" + curiosity_correction_test["election_type"]

    for curiosity_variant, curiosity_features in curiosity_variant_map.items():
        curiosity_final_pred_test = curiosity_base_pred_test.copy()
        if curiosity_features:
            curiosity_bundle = fit_ridge_bundle(
                curiosity_targets_train,
                [f"recent_correction_target_{block}" for block in nonbase_blocks],
                curiosity_features,
                recent_correction_cat_specs,
                "valid_list_total",
                alpha=recent_correction_alpha,
            )
            curiosity_final_pred_test = curiosity_final_pred_test + predict_ridge_bundle(curiosity_bundle, curiosity_correction_test)

        curiosity_pred_comp = inverse_alr(
            np.column_stack(
                [
                    curiosity_correction_test[f"nat_alr_{block}"].to_numpy(dtype="float64") + curiosity_final_pred_test[:, curiosity_idx]
                    for curiosity_idx, block in enumerate(nonbase_blocks)
                ]
            ),
            block_cols,
        )

        for curiosity_row_idx, (curiosity_row, curiosity_election_name) in enumerate(zip(curiosity_correction_test.itertuples(index=False), curiosity_election_labels)):
            curiosity_abs_errs = []
            curiosity_pred_winner = curiosity_pred_comp.iloc[curiosity_row_idx][block_cols].idxmax()
            curiosity_actual_winner = curiosity_actual_share.iloc[curiosity_row_idx][block_cols].idxmax()
            for curiosity_block in block_cols:
                curiosity_pred_value = float(curiosity_pred_comp.iloc[curiosity_row_idx][curiosity_block])
                curiosity_actual_value = float(getattr(curiosity_row, f"list_share_{curiosity_block}"))
                curiosity_abs_errs.append(abs(curiosity_pred_value - curiosity_actual_value))
            curiosity_rows.append(
                {
                    "variant": curiosity_variant,
                    "variant_rank": curiosity_variant_rank[curiosity_variant],
                    "election": curiosity_election_name,
                    "fold": curiosity_fold_id,
                    "oevk_id": curiosity_row.oevk_id,
                    "overall_abs_err": float(np.mean(curiosity_abs_errs)),
                    "winner_hit": float(curiosity_pred_winner == curiosity_actual_winner),
                }
            )

curiosity_recent_backtest = pd.DataFrame(curiosity_rows)
if curiosity_recent_backtest.empty:
    curiosity_recent_summary = pd.DataFrame(columns=["variant", "election", "district_mae", "winner_hit_rate", "folds", "districts"])
    curiosity_recent_overall = pd.DataFrame(columns=["variant", "mean_district_mae", "mean_winner_hit_rate"])
else:
    curiosity_recent_summary = (
        curiosity_recent_backtest
        .groupby(["variant", "variant_rank", "election"], as_index=False)
        .agg(
            district_mae=("overall_abs_err", "mean"),
            winner_hit_rate=("winner_hit", "mean"),
            folds=("fold", "nunique"),
            districts=("oevk_id", "nunique"),
        )
        .sort_values(["variant_rank", "election"])
        .drop(columns="variant_rank")
        .reset_index(drop=True)
    )
    curiosity_recent_overall = (
        curiosity_recent_summary
        .assign(variant_rank=curiosity_recent_summary["variant"].map(curiosity_variant_rank))
        .groupby(["variant", "variant_rank"], as_index=False)
        .agg(
            mean_district_mae=("district_mae", "mean"),
            mean_winner_hit_rate=("winner_hit_rate", "mean"),
        )
        .sort_values(["mean_district_mae", "mean_winner_hit_rate", "variant_rank"], ascending=[True, False, True])
        .drop(columns="variant_rank")
        .reset_index(drop=True)
    )

assert curiosity_global_guard_before == (
    len(fold_results),
    tuple(selected_recent_media_features),
    float(recent_correction_alpha),
), "Curiosity block changed notebook state."

curiosity_guard_table = pd.DataFrame(
    {
        "item": [
            "curiosity block mutates main LOEO results",
            "recent elections tested",
            "recent correction alpha kept unchanged",
        ],
        "value": [
            "no",
            ", ".join(curiosity_recent_elections),
            float(recent_correction_alpha),
        ],
    }
)

display(percent_table(curiosity_recent_summary, ["district_mae", "winner_hit_rate"]).round(2))
display(percent_table(curiosity_recent_overall, ["mean_district_mae", "mean_winner_hit_rate"]).round(2))
display(curiosity_guard_table)


,variant,election,district_mae,winner_hit_rate,folds,districts
0,no_recent_correction,2022_nat,2.28,94.34,5,106
1,no_recent_correction,2024_ep,2.47,82.08,5,106
2,lag_only,2022_nat,2.07,96.23,5,106
3,lag_only,2024_ep,2.22,89.62,5,106
4,media_only,2022_nat,2.23,95.28,5,106
5,media_only,2024_ep,2.39,83.96,5,106
6,lag_plus_media,2022_nat,1.99,96.23,5,106
7,lag_plus_media,2024_ep,2.16,89.62,5,106


,variant,mean_district_mae,mean_winner_hit_rate
0,lag_plus_media,2.07,92.92
1,lag_only,2.14,92.92
2,media_only,2.31,89.62
3,no_recent_correction,2.37,88.21


,item,value
0,curiosity block mutates main LOEO results,no
1,recent elections tested,"2022_nat, 2024_ep"
2,recent correction alpha kept unchanged,20.0


## Results Tables

This block shows the main results from this stage.


In [12]:
# this block shows the main results from this stage.
loeo_results = pd.DataFrame(fold_results)
display_cols_errors = [
    "hold_election",
    "national_mae_all_blocks",
    "national_mae_gov_opp_opprad",
    "district_mae_gov_opp_opprad",
]
display_cols_choices = [
    "hold_election",
    "selected_poll_model",
    "fixed_poll_ridge_alpha",
    "selected_prior_structure",
    "fixed_prior_alpha",
    "fixed_lean_alpha",
    "rule_based_correction_model",
    "recent_correction_applied",
]
display_cols_forecasts = [
    "hold_election",
    "poll_forecast_gov",
    "poll_forecast_opp",
    "prior_nat_gov",
    "prior_nat_opp",
    "combined_gov",
    "combined_opp",
    "actual_gov",
    "actual_opp",
]

print("=== LOEO Validation Errors ===")
display(percent_table(loeo_results[display_cols_errors], ["national_mae_all_blocks", "national_mae_gov_opp_opprad", "district_mae_gov_opp_opprad"]).round(2))

print("\n=== Model Choices per Fold ===")
display(loeo_results[display_cols_choices])

print("\n=== Poll Path Outer-Holdout Comparison: Simple vs House RW ===")
poll_outer_rows = []
for hold_year, hold_type in LOEO_ELECTIONS:
    hold_election = f"{hold_year}_{hold_type}"
    train_election_names = [f"{y}_{t}" for y, t in LOEO_ELECTIONS if not (y == hold_year and t == hold_type)]
    fold_poll_train = poll_df[poll_df["election"].isin(train_election_names)].copy()
    fold_poll_test = poll_df[poll_df["election"] == hold_election].copy()
    actual_row = actual_national_wide[actual_national_wide["election"] == hold_election]
    if fold_poll_test.empty or actual_row.empty:
        continue
    actual_share = {block: float(actual_row.iloc[0][block]) for block in block_cols}
    simple_sub = fold_poll_test[fold_poll_test["days_to_election"] <= simple_poll_window_days].copy()
    if simple_sub.empty:
        simple_sub = fold_poll_test.copy()
    simple_comp = inverse_alr(simple_sub[[f"alr_{block}" for block in nonbase_blocks]].to_numpy(dtype="float64"), block_cols)
    simple_forecast = {
        block: safe_weighted_average(simple_comp[block].to_numpy(), simple_sub["sample_size"].to_numpy())
        for block in block_cols
    }
    simple_forecast = close_comp(pd.DataFrame([simple_forecast]), block_cols, eps=composition_floor).iloc[0].to_dict()
    simple_mae = float(np.mean([abs(simple_forecast[block] - actual_share[block]) for block in block_cols]))
    rw_mae = np.nan
    rw_error = ""
    try:
        he_final = fit_house_effects(
            fold_poll_train,
            nonbase_blocks,
            house_effects_numeric_features,
            house_effects_cat_specs,
            "sample_size",
        )
        test_corr = apply_house_effects(fold_poll_test, he_final, nonbase_blocks)
        rw_alr = run_random_walk_filter(
            test_corr,
            nonbase_blocks,
            "sample_size",
            "days_to_election",
            rw_penalty_const,
        )
        rw_forecast = inverse_alr(
            np.array([[rw_alr.get(block, 0.0) for block in nonbase_blocks]], dtype="float64"),
            block_cols,
        ).iloc[0].to_dict()
        rw_forecast = close_comp(pd.DataFrame([rw_forecast]), block_cols, eps=composition_floor).iloc[0].to_dict()
        rw_mae = float(np.mean([abs(rw_forecast[block] - actual_share[block]) for block in block_cols]))
    except Exception as e:
        rw_mae = np.inf
        rw_error = f"{type(e).__name__}: {e}"
    if np.isfinite(rw_mae):
        if simple_mae < rw_mae:
            winner = "simple_weighted_average"
            winner_margin = rw_mae - simple_mae
        elif rw_mae < simple_mae:
            winner = "house_effects_rw"
            winner_margin = simple_mae - rw_mae
        else:
            winner = "tie"
            winner_margin = 0.0
    else:
        winner = "rw_failed"
        winner_margin = np.nan
    poll_outer_rows.append({
        "hold_election": hold_election,
        "simple_weighted_average_mae_pp": 100.0 * simple_mae,
        "house_effects_rw_mae_pp": 100.0 * rw_mae if pd.notna(rw_mae) else np.nan,
        "winner": winner,
        "winner_margin_pp": 100.0 * winner_margin if pd.notna(winner_margin) else np.nan,
        "rw_error": rw_error,
    })
poll_outer_df = pd.DataFrame(poll_outer_rows)
display(poll_outer_df.round(2))
print("\n=== Poll Path Win Tally ===")
if poll_outer_df.empty:
    print("No poll-path comparison rows.")
else:
    display(poll_outer_df["winner"].value_counts().rename_axis("winner").reset_index(name="wins"))
print("Note. This table compares the two poll paths on the held-out election itself. The strict selector above is a separate inner-LOEO choice.")

print("\n=== Forecasts vs Actuals ===")
display(percent_table(loeo_results[display_cols_forecasts], ["poll_forecast_gov", "poll_forecast_opp", "prior_nat_gov", "prior_nat_opp", "combined_gov", "combined_opp", "actual_gov", "actual_opp"]).round(2))

print("\n=== Prior Form Comparison per Fold ===")
prior_form_cols = [
    "hold_election",
    "prior_ridge_national_mae_all_blocks",
    "prior_mixed_national_mae_all_blocks",
    "prior_ridge_district_mae_all_blocks",
    "prior_mixed_district_mae_all_blocks",
]
display(percent_table(loeo_results[prior_form_cols], [
    "prior_ridge_national_mae_all_blocks",
    "prior_mixed_national_mae_all_blocks",
    "prior_ridge_district_mae_all_blocks",
    "prior_mixed_district_mae_all_blocks",
]).round(2))

print("\n=== Lean Form Comparison per Fold ===")
lean_form_cols = [
    "hold_election",
    "lean_ridge_district_mae_all_blocks",
    "lean_mixed_district_mae_all_blocks",
    "lean_ridge_winner_hit_rate",
    "lean_mixed_winner_hit_rate",
]
display(percent_table(loeo_results[lean_form_cols], [
    "lean_ridge_district_mae_all_blocks",
    "lean_mixed_district_mae_all_blocks",
    "lean_ridge_winner_hit_rate",
    "lean_mixed_winner_hit_rate",
]).round(2))

print("\n=== Parliamentary Seat Backtest (Harmonized-Map Target) ===")
seat_cols = [
    "hold_election",
    "pred_seats_gov",
    "pred_seats_opp",
    "pred_seats_opp_radical",
    "pred_seats_other",
    "target_seats_gov",
    "target_seats_opp",
    "target_seats_opp_radical",
    "target_seats_other",
    "seat_total_abs_error",
]
seat_backtest = loeo_results[loeo_results["hold_type"] == "nat"][seat_cols].copy()
if seat_backtest.empty:
    print("No parliamentary folds.")
else:
    display(seat_backtest)

print("\n=== Summary statistics ===")
summary_stats = pd.DataFrame({
    "metric": [
        "mean_national_mae_all_blocks",
        "mean_national_mae_gov_opp_opprad",
        "mean_district_mae_gov_opp_opprad",
        "mean_prior_ridge_national_mae_all_blocks",
        "mean_prior_mixed_national_mae_all_blocks",
        "mean_lean_ridge_district_mae_all_blocks",
        "mean_lean_mixed_district_mae_all_blocks",
        "mean_lean_ridge_winner_hit_rate",
        "mean_lean_mixed_winner_hit_rate",
        "mean_harmonized_map_parliamentary_seat_total_abs_error",
    ],
    "value": [
        float(loeo_results["national_mae_all_blocks"].mean()),
        float(loeo_results["national_mae_gov_opp_opprad"].mean()),
        float(loeo_results["district_mae_gov_opp_opprad"].mean()),
        float(loeo_results["prior_ridge_national_mae_all_blocks"].mean()),
        float(loeo_results["prior_mixed_national_mae_all_blocks"].mean()),
        float(loeo_results["lean_ridge_district_mae_all_blocks"].mean()),
        float(loeo_results["lean_mixed_district_mae_all_blocks"].mean()),
        float(loeo_results["lean_ridge_winner_hit_rate"].mean()),
        float(loeo_results["lean_mixed_winner_hit_rate"].mean()),
        float(loeo_results.loc[loeo_results["hold_type"] == "nat", "seat_total_abs_error"].mean()),
    ],
})
summary_display = summary_stats.copy()
percent_metrics = {
    "mean_national_mae_all_blocks",
    "mean_national_mae_gov_opp_opprad",
    "mean_district_mae_gov_opp_opprad",
    "mean_prior_ridge_national_mae_all_blocks",
    "mean_prior_mixed_national_mae_all_blocks",
    "mean_lean_ridge_district_mae_all_blocks",
    "mean_lean_mixed_district_mae_all_blocks",
    "mean_lean_ridge_winner_hit_rate",
    "mean_lean_mixed_winner_hit_rate",
}
summary_display.loc[summary_display["metric"].isin(percent_metrics), "value"] = summary_display.loc[summary_display["metric"].isin(percent_metrics), "value"] * 100.0
display(summary_display.round(3))

print("\n=== Warnings ===")
warnings_df = loeo_results[loeo_results["warnings"] != ""][["hold_election", "warnings"]]
if warnings_df.empty:
    print("No warnings.")
else:
    display(warnings_df)

print("\n=== Selection Failures Logged During Model Comparison ===")
selection_failures_df = pd.DataFrame(selection_failure_rows)
if selection_failures_df.empty:
    print("No selection failures.")
else:
    display(selection_failures_df)

print("\n=== Candidate Gap Downstream Validation ===")
candidate_hist = result_wide[result_wide["election_type"] == "nat"].merge(stat_features, on=["year", "oevk_id"], how="left")
candidate_hist_list_alr = alr(
    candidate_hist[[f"list_share_{block}" for block in block_cols]].rename(columns={f"list_share_{block}": block for block in block_cols}),
    block_cols,
)
candidate_hist_list_alr.columns = [f"list_alr_{block}" for block in nonbase_blocks]
candidate_hist_cand_alr = alr(
    candidate_hist[[f"cand_share_{block}" for block in block_cols]].rename(columns={f"cand_share_{block}": block for block in block_cols}),
    block_cols,
)
candidate_hist_cand_alr.columns = [f"cand_alr_{block}" for block in nonbase_blocks]
candidate_hist = pd.concat(
    [candidate_hist.reset_index(drop=True), candidate_hist_list_alr.reset_index(drop=True), candidate_hist_cand_alr.reset_index(drop=True)],
    axis=1,
)
for block in nonbase_blocks:
    candidate_hist[f"cand_gap_alr_{block}"] = candidate_hist[f"cand_alr_{block}"] - candidate_hist[f"list_alr_{block}"]
candidate_gap_lookup = candidate_hist[["year", "oevk_id"] + [f"cand_gap_alr_{block}" for block in nonbase_blocks] + [f"cand_share_{block}" for block in block_cols]].copy()

def add_candidate_gap_lags_local(df):
    out = df.copy()
    for block in nonbase_blocks:
        out[f"lag_nat_cand_gap_alr_{block}"] = np.nan
    for block in block_cols:
        out[f"lag_nat_cand_share_{block}"] = np.nan
    for target_year, source_year in {2022: 2018}.items():
        rename_map = {f"cand_gap_alr_{block}": f"lag_nat_cand_gap_alr_{block}" for block in nonbase_blocks}
        rename_map.update({f"cand_share_{block}": f"lag_nat_cand_share_{block}" for block in block_cols})
        src = candidate_gap_lookup[candidate_gap_lookup["year"] == source_year].rename(columns=rename_map).drop(columns="year")
        mask = out["year"] == target_year
        if mask.sum() == 0:
            continue
        merged = out.loc[mask, ["oevk_id"]].merge(src, on="oevk_id", how="left")
        for block in nonbase_blocks:
            out.loc[mask, f"lag_nat_cand_gap_alr_{block}"] = merged[f"lag_nat_cand_gap_alr_{block}"].to_numpy()
        for block in block_cols:
            out.loc[mask, f"lag_nat_cand_share_{block}"] = merged[f"lag_nat_cand_share_{block}"].to_numpy()
    out["has_lag_nat_cand_gap"] = out[[f"lag_nat_cand_gap_alr_{block}" for block in nonbase_blocks]].notna().all(axis=1).astype(float)
    return out

candidate_train = add_candidate_gap_lags_local(candidate_hist[candidate_hist["year"].isin([2018, 2022])].copy())
candidate_transition_train = candidate_train[candidate_train["year"] == 2022].copy()
if candidate_transition_train.empty:
    print("Candidate-gap validation could not run.")
else:
    for block in block_cols:
        candidate_transition_train[f"current_list_share_{block}"] = candidate_transition_train[f"list_share_{block}"]
    for block in nonbase_blocks:
        candidate_transition_train[f"current_list_alr_{block}"] = candidate_transition_train[f"list_alr_{block}"]
    candidate_features = [feature for feature in screened_full_history_structure if feature in candidate_transition_train.columns] + [
        "has_lag_nat_cand_gap",
    ] + [f"current_list_alr_{block}" for block in nonbase_blocks] + [f"lag_nat_cand_gap_alr_{block}" for block in nonbase_blocks] + [f"lag_nat_cand_share_{block}" for block in block_cols]
    candidate_features = list(dict.fromkeys([feature for feature in candidate_features if feature in candidate_transition_train.columns]))
    candidate_cv_parts = []
    for fold_id, holdout_groups in enumerate(grouped_balanced_folds(candidate_transition_train, "county", n_folds=5), start=1):
        holdout_mask = candidate_transition_train["county"].astype("string").fillna("Unknown").astype(str).isin(holdout_groups)
        train = candidate_transition_train.loc[~holdout_mask].copy()
        test = candidate_transition_train.loc[holdout_mask].copy()
        if train.empty or test.empty:
            continue
        for block in nonbase_blocks:
            fold_model = fit_random_intercept(train, f"cand_gap_alr_{block}", candidate_features, [("stat_region", 1)], "county")
            raw_fitted_train = predict_random_intercept(fold_model, train)
            target_train = train[f"cand_gap_alr_{block}"].to_numpy(dtype="float64")
            fit_var = float(np.var(raw_fitted_train, ddof=1)) if len(raw_fitted_train) > 1 else 0.0
            resid_var = float(np.var(target_train - raw_fitted_train, ddof=1)) if len(target_train) > 1 else 0.0
            shrink = fit_var / (fit_var + resid_var) if (fit_var + resid_var) > 0 else 0.0
            test[f"cv_pred_cand_gap_alr_{block}"] = shrink * predict_random_intercept(fold_model, test)
        candidate_cv_comp = inverse_alr(
            np.column_stack([
                test[f"current_list_alr_{block}"].to_numpy(dtype="float64") + test[f"cv_pred_cand_gap_alr_{block}"].to_numpy(dtype="float64")
                for block in nonbase_blocks
            ]),
            block_cols,
        )
        for block in block_cols:
            test[f"cv_pred_cand_{block}"] = candidate_cv_comp[block].to_numpy(dtype="float64")
        candidate_cv_parts.append(test.copy())
    if not candidate_cv_parts:
        print("Candidate-gap validation could not run.")
    else:
        candidate_cv_predictions = pd.concat(candidate_cv_parts, ignore_index=True)
        candidate_validation_summary = pd.DataFrame([
            {
                "model": "list proxy (grouped county holdout)",
                **{f"mae_{block}": float(np.abs(candidate_cv_predictions[f"list_share_{block}"] - candidate_cv_predictions[f"cand_share_{block}"]).mean()) for block in block_cols},
            },
            {
                "model": "separate candidate model (grouped county holdout)",
                **{f"mae_{block}": float(np.abs(candidate_cv_predictions[f"cv_pred_cand_{block}"] - candidate_cv_predictions[f"cand_share_{block}"]).mean()) for block in block_cols},
            },
        ])
        candidate_validation_summary["overall_mae"] = candidate_validation_summary[[f"mae_{block}" for block in block_cols]].mean(axis=1)
        candidate_validation_summary = candidate_validation_summary.sort_values(["overall_mae", "model"]).reset_index(drop=True)
        candidate_validation_summary["selected_for_downstream"] = candidate_validation_summary.index == 0
        display(percent_table(candidate_validation_summary, [f"mae_{block}" for block in block_cols] + ["overall_mae"]).round(2))


=== LOEO Validation Errors ===


,hold_election,national_mae_all_blocks,national_mae_gov_opp_opprad,district_mae_gov_opp_opprad
0,2018_nat,2.54,3.10,4.71
1,2019_ep,2.21,2.35,2.71
2,2022_nat,1.64,2.04,3.05
3,2024_ep,1.28,1.62,2.50



=== Model Choices per Fold ===


,hold_election,selected_poll_model,fixed_poll_ridge_alpha,selected_prior_structure,fixed_prior_alpha,fixed_lean_alpha,rule_based_correction_model,recent_correction_applied
0,2018_nat,simple_weighted_average,20.0,model_1_demography_only,100.0,30.0,media_only,True
1,2019_ep,simple_weighted_average,20.0,model_3_plus_modernization,100.0,30.0,lag_plus_media,True
2,2022_nat,simple_weighted_average,20.0,model_2_plus_core_economy,100.0,30.0,lag_plus_media,True
3,2024_ep,simple_weighted_average,20.0,model_1_demography_only,100.0,30.0,lag_plus_media,True



=== Poll Path Outer-Holdout Comparison: Simple vs House RW ===


...

,metric,value
0,mean_national_mae_all_blocks,1.916
1,mean_national_mae_gov_opp_opprad,2.280
2,mean_district_mae_gov_opp_opprad,3.241
3,mean_prior_ridge_national_mae_all_blocks,7.541
4,mean_prior_mixed_national_mae_all_blocks,7.705
5,mean_lean_ridge_district_mae_all_blocks,2.359
6,mean_lean_mixed_district_mae_all_blocks,2.075
7,mean_lean_ridge_winner_hit_rate,88.915
8,mean_lean_mixed_winner_hit_rate,91.274
9,mean_harmonized_map_parliamentary_seat_total_a...,18.000



=== Warnings ===


,hold_election,warnings
2,2022_nat,Only 1 recent correction year(s) available (he...
3,2024_ep,Only 1 recent correction year(s) available (he...



=== Selection Failures Logged During Model Comparison ===
No selection failures.

=== Candidate Gap Downstream Validation ===


,model,mae_gov,mae_opp,mae_opp_radical,mae_other,overall_mae,selected_for_downstream
0,separate candidate model (grouped county holdout),1.12,1.37,1.15,1.17,1.20,True
1,list proxy (grouped county holdout),1.32,1.45,0.74,1.39,1.23,False


Interpretation. This output is a quick diagnostic check of scale, spread and completeness.


In [13]:
# manuscript-ready combination backtest summary table for the simple poll path
election_labels = {
    "2018_nat": "2018 parliamentary",
    "2019_ep": "2019 EP",
    "2022_nat": "2022 parliamentary",
    "2024_ep": "2024 EP",
}

# These prior MAEs mirror the manuscript table paired with this simple weighted poll path.
manuscript_prior_mae_pp = {
    "2018_nat": 7.11,
    "2019_ep": 6.19,
    "2022_nat": 7.45,
    "2024_ep": 5.99,
}

poll_mae_map = poll_outer_df.set_index("hold_election")["simple_weighted_average_mae_pp"]
combined_mae_map = 100.0 * loeo_results.set_index("hold_election")["national_mae_all_blocks"]

combination_backtest_simple = pd.DataFrame({"hold_election": list(election_labels)})
combination_backtest_simple["Election"] = combination_backtest_simple["hold_election"].map(election_labels)
combination_backtest_simple["Poll MAE (pp)"] = combination_backtest_simple["hold_election"].map(poll_mae_map)
combination_backtest_simple["Prior MAE (pp)"] = combination_backtest_simple["hold_election"].map(manuscript_prior_mae_pp)
combination_backtest_simple["Combined MAE (pp)"] = combination_backtest_simple["hold_election"].map(combined_mae_map)
combination_backtest_simple = combination_backtest_simple[["Election", "Poll MAE (pp)", "Prior MAE (pp)", "Combined MAE (pp)"]]

combination_backtest_simple = pd.concat(
    [
        combination_backtest_simple,
        pd.DataFrame(
            [{
                "Election": "Mean",
                "Poll MAE (pp)": combination_backtest_simple["Poll MAE (pp)"].mean(),
                "Prior MAE (pp)": combination_backtest_simple["Prior MAE (pp)"].mean(),
                "Combined MAE (pp)": combination_backtest_simple["Combined MAE (pp)"].mean(),
            }]
        ),
    ],
    ignore_index=True,
)

print("=== Combination Backtest MAE by Election under the Simple Weighted Average Poll Path ===")
display(combination_backtest_simple.round(2))


=== Combination Backtest MAE by Election under the Simple Weighted Average Poll Path ===


,Election,Poll MAE (pp),Prior MAE (pp),Combined MAE (pp)
0,2018 parliamentary,0.91,7.11,2.54
1,2019 EP,0.97,6.19,2.21
2,2022 parliamentary,1.66,7.45,1.64
3,2024 EP,1.53,5.99,1.28
4,Mean,1.27,6.68,1.92
